## Dependencies and Setup

Cette partie prépare une exécution identique autant que possible en local, sur Colab ou sur Kaggle : dépendances, imports, tracking MLflow, reproductibilité, ressources NLTK, périphérique de calcul et emplacement des données.

### Dependencies

Cette section détecte l'environnement d'exécution et installe uniquement les dépendances qui lui sont propres. Kaggle conserve ses builds TensorFlow et PyTorch compatibles GPU ; en local, l'environnement complet est déclaré dans `requirements-local.txt`.

### Imports

Cette section importe les bibliothèques utilisées par les familles de modèles actuelles et par les futures extensions du benchmark. Les imports optionnels restent visibles afin de rendre explicites les dépendances du notebook.

In [11]:
from pathlib import Path

# Kaggle uploads the notebook itself, not sibling requirements files. Keep the
# runtime dependency contract explicit and self-contained here. TensorFlow and
# PyTorch are intentionally omitted: Kaggle provides CUDA-compatible builds.
if Path('/kaggle/working').exists():
    import subprocess
    import sys

    KAGGLE_RUNTIME_PACKAGES = [
        'mlflow[databricks]>=3.13,<4',
        'python-dotenv>=1.0,<2',
        'contractions>=0.1.73,<1',
        'emoji>=2.0,<3',
        'gensim>=4.3,<5',
        'matplotlib>=3.8,<4',
        'nltk>=3.8,<4',
        'numpy>=1.26,<3',
        'optuna>=4.0,<5',
        'pandas>=2.0,<3',
        'scikit-learn>=1.4,<2',
        'seaborn>=0.13,<1',
        'tqdm>=4.66,<5',
        'transformers>=4.40,<6',
    ]
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade-strategy', 'only-if-needed', *KAGGLE_RUNTIME_PACKAGES],
        check=True,
    )
elif Path('/content').exists(): 
    from google.colab import drive
    drive.mount('/content/drive') # Connect to Google Drive if the notebook is run on Colab.
    !pip install emoji mlflow optuna contractions gensim transformers # Install missing dependencies if run on Colab.
else: 
    pass # Install requirements-local.txt before running the notebook locally.


In [ ]:
import os
import gc
import re
import sys
import time
import tempfile
import copy
import html
import json
import random
from datetime import datetime
from collections import Counter, OrderedDict

import emoji
import mlflow
import mlflow.pytorch
import mlflow.sklearn
import mlflow.tensorflow
import nltk
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
from contractions import fix
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow import keras
from tensorflow.keras.layers import Dense, Embedding, GRU, LSTM
from tensorflow.keras.metrics import FBetaScore
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tqdm import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, fbeta_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

### Tracking, provenance and reproducibility utilities

Cette section sélectionne le backend MLflow local ou Databricks, lit les paramètres non sensibles et les secrets au bon endroit, prépare le manifeste du dataset et centralise la seed utilisée par toutes les étapes du benchmark.

In [13]:
from dataclasses import field, dataclass
from urllib.parse import urlparse

DATABRICKS_REQUIRED_SETTINGS = ('DATABRICKS_HOST', 'DATABRICKS_TOKEN', 'MLFLOW_EXPERIMENT_ID')
DEFAULT_DATABRICKS_TRACKING_URI = 'databricks'
DEFAULT_LOCAL_EXPERIMENT_NAME = 'sentiment140-benchmark-v1-local'
VALID_TRACKING_MODES = {'local', 'databricks'}
PROJECT_SEED_ENV_VAR = 'PROJECT_SEED'
DEFAULT_PROJECT_SEED = 42
MAX_NUMPY_SEED = 2**32 - 1
SENTIMENT140_TRAINING_FILENAME = 'training.1600000.processed.noemoticon.csv'
SENTIMENT140_ENCODING = 'cp1252'



In [14]:
class TrackingConfigurationError(RuntimeError):
    """Raised when MLflow tracking cannot be configured safely."""


class ReproducibilityConfigurationError(ValueError):
    """Raised when the configured project seed is invalid."""


class DatasetProvenanceError(ValueError):
    """Raised when the Sentiment140 source file is not the expected corpus."""


@dataclass(frozen=True)
class TrackingConfig:
    """Describe the resolved MLflow backend without exposing the access token."""
    backend: str
    experiment_id: str
    experiment_name: str
    tracking_uri: str
    host: str | None = None
    token: str | None = field(default=None, repr=False)


@dataclass(frozen=True)
class TrackingSmokeTestResult:
    """Record the identifiers returned by a successful tracking smoke test."""
    backend: str
    experiment_id: str
    run_id: str
    artifact_path: str


@dataclass(frozen=True)
class ReproducibilityConfig:
    """Record the seed and deterministic-operation status for one execution."""
    seed: int
    tensorflow_status: str
    pytorch_status: str
    deterministic_operations_requested: bool

    def as_mlflow_params(self):
        """Return reproducibility metadata in MLflow-compatible form."""
        return {
            'project_seed': self.seed,
            'deterministic_operations_requested': str(self.deterministic_operations_requested).lower(),
            'tensorflow_seed_status': self.tensorflow_status,
            'pytorch_seed_status': self.pytorch_status,
        }


@dataclass(frozen=True)
class DatasetManifest:
    """Describe the expected dataset without logging its tweet contents."""
    filename: str
    byte_size: int
    encoding: str = SENTIMENT140_ENCODING

    def as_mlflow_params(self):
        """Return lightweight dataset provenance metadata for MLflow."""
        return {
            'dataset_filename': self.filename,
            'dataset_byte_size': self.byte_size,
            'dataset_encoding': self.encoding,
        }


def _is_kaggle_runtime():
    """Return whether the current kernel is running in Kaggle."""
    return Path('/kaggle/working').exists()


def _read_kaggle_secret(name):
    """Read one Kaggle Secret, returning None when it is unavailable."""
    if not _is_kaggle_runtime():
        return None
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        return None
    return value.strip() if value else None


def _read_setting(name):
    """Resolve one setting from process variables first, then Kaggle Secrets."""
    value = os.environ.get(name) or _read_kaggle_secret(name)
    return value.strip() if value and value.strip() else None


def _load_local_dotenv():
    """Load a local .env file without overriding explicit shell variables."""
    if _is_kaggle_runtime():
        return
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
    except ImportError:
        pass


def _resolve_tracking_mode():
    """Choose the permitted MLflow backend for the current runtime."""
    _load_local_dotenv()
    requested_mode = _read_setting('MLFLOW_TRACKING_MODE')
    if requested_mode:
        mode = requested_mode.lower()
        if mode not in VALID_TRACKING_MODES:
            raise TrackingConfigurationError('MLFLOW_TRACKING_MODE must be local or databricks.')
    elif _is_kaggle_runtime():
        mode = 'databricks'
    else:
        mode = 'databricks' if all(_read_setting(name) for name in DATABRICKS_REQUIRED_SETTINGS) else 'local'
    if _is_kaggle_runtime() and mode != 'databricks':
        raise TrackingConfigurationError('Kaggle benchmark runs must use Databricks MLflow tracking.')
    return mode


def _configure_local_tracking():
    """Create or reuse the local SQLite-backed MLflow experiment."""
    from mlflow import MlflowClient

    # local mlflow artifact directory
    local_directory = _read_setting('MLFLOW_LOCAL_DIR')
    artifact_root = Path(local_directory).expanduser() if local_directory else Path.cwd().resolve() / 'mlruns'
    if not artifact_root.is_absolute():
        artifact_root = Path.cwd().resolve() / artifact_root
    artifact_root = artifact_root.resolve()
    artifact_root.mkdir(parents=True, exist_ok=True)

    # mlflow experiment name and local database
    experiment_name = _read_setting('MLFLOW_EXPERIMENT_NAME') or DEFAULT_LOCAL_EXPERIMENT_NAME
    tracking_uri = f"sqlite:///{(artifact_root / 'mlflow.db').as_posix()}"
    mlflow.set_tracking_uri(tracking_uri)

    # mlflow client experiment connection
    client = MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = client.create_experiment(experiment_name, artifact_location=(artifact_root / 'artifacts').as_uri())
        experiment = client.get_experiment(experiment_id)
    mlflow.set_experiment(experiment.name)
    os.environ.update({
        'MLFLOW_TRACKING_URI': tracking_uri,
        'MLFLOW_EXPERIMENT_ID': experiment.experiment_id,
        'MLFLOW_EXPERIMENT_NAME': experiment.name,
    })
    return TrackingConfig('local', experiment.experiment_id, experiment.name, tracking_uri)


def configure_tracking():
    """Configure and validate either local MLflow or remote Databricks tracking."""
    if _resolve_tracking_mode() == 'local':
        return _configure_local_tracking()

    values = {name: _read_setting(name) for name in DATABRICKS_REQUIRED_SETTINGS}
    missing = [name for name, value in values.items() if not value]
    if missing:
        raise TrackingConfigurationError('Add these Kaggle Secrets or environment variables: ' + ', '.join(missing))
    host = values['DATABRICKS_HOST'].rstrip('/')
    parsed_host = urlparse(host)
    if parsed_host.scheme != 'https' or not parsed_host.netloc:
        raise TrackingConfigurationError('DATABRICKS_HOST must be a complete HTTPS workspace URL.')
    experiment_id = values['MLFLOW_EXPERIMENT_ID']
    if not experiment_id.isdigit():
        raise TrackingConfigurationError('MLFLOW_EXPERIMENT_ID must be a numeric MLflow experiment ID.')
    tracking_uri = _read_setting('MLFLOW_TRACKING_URI') or DEFAULT_DATABRICKS_TRACKING_URI
    os.environ.update({
        'DATABRICKS_HOST': host,
        'DATABRICKS_TOKEN': values['DATABRICKS_TOKEN'],
        'MLFLOW_TRACKING_URI': tracking_uri,
        'MLFLOW_EXPERIMENT_ID': experiment_id,
    })
    mlflow.set_tracking_uri(tracking_uri)
    experiment = mlflow.MlflowClient().get_experiment(experiment_id)
    if experiment is None or experiment.lifecycle_stage != 'active':
        raise TrackingConfigurationError(f'MLflow experiment {experiment_id} is unavailable or inactive.')
    return TrackingConfig('databricks', experiment_id, experiment.name, tracking_uri, host, values['DATABRICKS_TOKEN'])


def run_tracking_smoke_test(config=None):
    """Log and read back a minimal parameter, metric and text artifact."""
    config = config or configure_tracking()
    artifact_name = 'tracking_smoke_test.txt'
    timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
    with mlflow.start_run(
        experiment_id=config.experiment_id,
        run_name=f'tracking-smoke-test-{timestamp}',
        tags={
            'run_type': 'tracking_smoke_test',
            'tracking_backend': config.backend,
            'execution_environment': 'kaggle' if _is_kaggle_runtime() else 'local',
        },
    ) as active_run:
        mlflow.log_param('smoke_test', 'true')
        mlflow.log_metric('tracking_connection', 1.0)
        with tempfile.TemporaryDirectory(prefix='mlflow-smoke-') as temporary_directory:
            artifact_path = Path(temporary_directory) / artifact_name
            artifact_path.write_text('MLflow tracking smoke test completed successfully.\n', encoding='utf-8')
            mlflow.log_artifact(str(artifact_path))
        run_id = active_run.info.run_id
    artifact_names = {artifact.path for artifact in mlflow.MlflowClient().list_artifacts(run_id)}
    if artifact_name not in artifact_names:
        raise TrackingConfigurationError('The smoke-test run completed but its artifact could not be read back.')
    return TrackingSmokeTestResult(config.backend, config.experiment_id, run_id, artifact_name)


def _read_kaggle_seed():
    """Return the optional project seed stored as a Kaggle Secret."""
    return _read_kaggle_secret(PROJECT_SEED_ENV_VAR)


def get_project_seed():
    """Resolve and validate the single integer seed used by the benchmark."""
    raw_seed = os.environ.get(PROJECT_SEED_ENV_VAR) or _read_kaggle_seed() or str(DEFAULT_PROJECT_SEED)
    try:
        seed = int(raw_seed.strip())
    except ValueError as error:
        raise ReproducibilityConfigurationError(f'{PROJECT_SEED_ENV_VAR} must be an integer between 0 and {MAX_NUMPY_SEED}.') from error
    if not 0 <= seed <= MAX_NUMPY_SEED:
        raise ReproducibilityConfigurationError(f'{PROJECT_SEED_ENV_VAR} must be between 0 and {MAX_NUMPY_SEED}.')
    return seed


def configure_reproducibility(seed=None, deterministic_operations=True):
    """Seed Python, NumPy and available ML frameworks for one benchmark run."""
    seed = get_project_seed() if seed is None else seed
    if not 0 <= seed <= MAX_NUMPY_SEED:
        raise ReproducibilityConfigurationError(f'seed must be between 0 and {MAX_NUMPY_SEED}.')
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    tensorflow_status = 'not_installed'
    try:
        import tensorflow as tensorflow_runtime
        tensorflow_runtime.keras.utils.set_random_seed(seed)
        if deterministic_operations:
            try:
                tensorflow_runtime.config.experimental.enable_op_determinism()
                tensorflow_status = 'seeded_and_deterministic_ops_enabled'
            except (AttributeError, RuntimeError):
                tensorflow_status = 'seeded_deterministic_ops_unavailable'
        else:
            tensorflow_status = 'seeded'
    except ImportError:
        pass

    pytorch_status = 'not_installed'
    try:
        import torch as torch_runtime
        torch_runtime.manual_seed(seed)
        if torch_runtime.cuda.is_available():
            torch_runtime.cuda.manual_seed_all(seed)
        if deterministic_operations:
            torch_runtime.backends.cudnn.benchmark = False
            torch_runtime.backends.cudnn.deterministic = True
            torch_runtime.use_deterministic_algorithms(True, warn_only=True)
            pytorch_status = 'seeded_and_deterministic_ops_requested'
        else:
            pytorch_status = 'seeded'
    except ImportError:
        pass
    return ReproducibilityConfig(seed, tensorflow_status, pytorch_status, deterministic_operations)

In [15]:
# The runtime helpers are defined in the notebook Utils section so this cell
# works both from a local checkout and from a Kaggle CLI-published notebook.
cwd = Path.cwd().resolve()
tracking_config = configure_tracking()
reproducibility_config = configure_reproducibility()
PROJECT_SEED = reproducibility_config.seed
RANDOM_STATE = PROJECT_SEED  # Backward-compatible name used by sampling and splitting below.

print('MLflow backend:', tracking_config.backend)
print('MLflow experiment:', tracking_config.experiment_name, f'(ID: {tracking_config.experiment_id})')
print('Project seed:', PROJECT_SEED)
print('TensorFlow reproducibility:', reproducibility_config.tensorflow_status)
print('PyTorch reproducibility:', reproducibility_config.pytorch_status)

MLflow backend: databricks
MLflow experiment: /Shared/sentiment140-benchmark-v1 (ID: 3829536073772222)
Project seed: 42
TensorFlow reproducibility: not_installed
PyTorch reproducibility: not_installed


### Device (GPU/CPU) Setup

Cette section télécharge seulement les ressources NLTK absentes, détecte les accélérateurs disponibles et choisit un périphérique PyTorch compatible. Elle affiche ensuite un diagnostic utile avant un entraînement.

In [ ]:
FLAGS = re.MULTILINE | re.DOTALL

def ensure_nltk_resource(resource_path, download_name=None):
    """Download an NLTK resource only when it is not already available locally."""
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name or resource_path.split('/')[-1], quiet=True)

ensure_nltk_resource('tokenizers/punkt', 'punkt')
ensure_nltk_resource('tokenizers/punkt_tab', 'punkt_tab')
ensure_nltk_resource('corpora/stopwords', 'stopwords')
ensure_nltk_resource('corpora/wordnet', 'wordnet')
ensure_nltk_resource('corpora/omw-1.4', 'omw-1.4')

def resolve_torch_device(min_cuda_capability=(7, 0)):
    """Return the PyTorch device while falling back to CPU when the available GPU is too old
    for the installed Torch build."""
    if not torch.cuda.is_available():
        return 'cpu', None, 'CUDA is not available for PyTorch in this environment.'

    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)
    if gpu_capability < min_cuda_capability:
        message = (
            f'PyTorch will run on CPU because the available GPU ({gpu_name}, sm_{gpu_capability[0]}{gpu_capability[1]}) '
            f'is older than the minimum capability supported by this Torch build (sm_{min_cuda_capability[0]}{min_cuda_capability[1]}).'
        )
        return 'cpu', gpu_name, message

    return 'cuda', gpu_name, 'PyTorch can use the GPU in this environment.'

tf_gpu_available = bool(tf.config.list_physical_devices('GPU'))
device, torch_gpu_name, torch_device_message = resolve_torch_device()

print('TensorFlow version:', tf.__version__)
print('PyTorch version:', torch.__version__)
print('TensorFlow GPU available:', tf_gpu_available)
print('PyTorch CUDA available:', torch.cuda.is_available())
if torch_gpu_name is not None:
    print('Detected GPU for PyTorch:', torch_gpu_name)
print('PyTorch device used by the notebook:', device)
print(torch_device_message)

### Data location and Working Directory

Cette section localise le fichier Sentiment140 selon l'environnement, vérifie son nom et son existence, puis prépare les métadonnées légères de provenance utilisées dans MLflow.

In [16]:
def verify_sentiment140_training_file(path):
    """Validate the expected Sentiment140 filename and return its manifest."""
    dataset_path = Path(path)
    if dataset_path.name != SENTIMENT140_TRAINING_FILENAME:
        raise DatasetProvenanceError(f'Expected {SENTIMENT140_TRAINING_FILENAME}, received {dataset_path.name}.')
    if not dataset_path.is_file():
        raise DatasetProvenanceError(f'Dataset file does not exist: {dataset_path}')
    return DatasetManifest(dataset_path.name, dataset_path.stat().st_size)

In [17]:
RAW_DATA_FILENAME = 'training.1600000.processed.noemoticon.csv'

# Detect the working environment and declare candidate directories for the raw dataset.
if Path('/kaggle/working').exists():
    RUN_ENV = 'kaggle'
    data_candidates = [
        Path('/kaggle/input/sentiment140'),
        Path('/kaggle/input/datasets/mouhamadoudiagne/sentiment140'),
        Path('/kaggle/input')
    ]
elif Path('/content').exists():
    RUN_ENV = 'colab'
    data_candidates = [
        Path('/content/drive/MyDrive/AI Engineer OC/Projet 7/data/sentiment140'),
        Path('/content/drive/MyDrive/AI Engineer OC/Projet 7'),
        Path('/content')
    ]
else:
    RUN_ENV = 'local'
    data_candidates = [
        cwd / 'data' / 'sentiment140',
        cwd.parent / 'data' / 'sentiment140',
        cwd / 'data',
        cwd.parent / 'data'
    ]

# Expand each candidate so the notebook can work whether the candidate is the dataset root
# itself or a parent directory that contains a `sentiment140` folder.
raw_data_candidates = []
for candidate_dir in data_candidates:
    raw_data_candidates.append(candidate_dir / RAW_DATA_FILENAME)
    raw_data_candidates.append(candidate_dir / 'sentiment140' / RAW_DATA_FILENAME)

RAW_DATA_PATH = next((path for path in raw_data_candidates if path.exists()), None)
if RAW_DATA_PATH is None:
    searched_paths = '\n'.join(f'- {path}' for path in raw_data_candidates)
    raise FileNotFoundError(
        f'Raw dataset not found. The notebook looked for `{RAW_DATA_FILENAME}` in:\n{searched_paths}'
    )

DATA_PATH = RAW_DATA_PATH.parent
print('Working directory :', cwd)
print('Execution environment :', RUN_ENV)
print('Data directory :', DATA_PATH)
print('Raw dataset path :', RAW_DATA_PATH)
print('Raw dataset exists :', RAW_DATA_PATH.exists())

dataset_manifest = verify_sentiment140_training_file(RAW_DATA_PATH)
RUN_PROVENANCE_PARAMS = {
    **reproducibility_config.as_mlflow_params(),
    **dataset_manifest.as_mlflow_params(),
}
print('Dataset size (bytes):', dataset_manifest.byte_size)

Working directory : C:\Users\Dell\Desktop\OC_AI_Engineer\projet_7\notebooks
Execution environment : local
Data directory : C:\Users\Dell\Desktop\OC_AI_Engineer\projet_7\data\sentiment140
Raw dataset path : C:\Users\Dell\Desktop\OC_AI_Engineer\projet_7\data\sentiment140\training.1600000.processed.noemoticon.csv
Raw dataset exists : True
Dataset size (bytes): 238803811


## Benchmark Utils

Cette section regroupe les briques réutilisables du benchmark : prétraitement versionné, configuration des candidates, métriques métier, sélection sur validation, artifacts MLflow et utilitaires neuronaux. Les deux familles de modèles s'appuient sur ce contrat commun.

### Global Variables

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from time import perf_counter
from typing import Any, Callable, Iterable


NEGATIVE_LABEL = 0
POSITIVE_LABEL = 1
BADBUZZ_REFERENCE_VOLUME = 1000
FBETA_BETA = 3.0
MINIMUM_VALIDATION_ACCURACY = 0.60
DEFAULT_DECISION_THRESHOLD = 0.5
SOCIAL_TEXT_V1_DEFAULT_PARAMETERS = {
    'normalize_whitespace': True,
    'decode_html_entities': True,
    'expand_contractions': True,
    'replace_urls': True,
    'replace_emails': True,
    'replace_mentions': True,
    'expand_hashtags': True,
    'demojize_emojis': True,
    'replace_ascii_emoticons': True,
    'reduce_elongated_tokens': True,
    'replace_numbers': True,
    'encode_sentiment_punctuation': True,
    'remove_digits': False,
    'remove_punctuation': False,
}


_EMAIL_PATTERN = re.compile(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b')
_URL_PATTERN = re.compile(r'(?i)\b(?:https?://|www\.)\S+')
_MENTION_PATTERN = re.compile(r'(?<!\w)@\w+')
_HASHTAG_PATTERN = re.compile(r'(?<!\w)#(\w+)')
_NUMBER_PATTERN = re.compile(r'\b\d+(?:[.,]\d+)?\b')
_PUNCTUATION_PATTERN = re.compile(r'[^\w\s]')
_ASCII_EMOTICON_REPLACEMENTS = (
    (re.compile(r"(?i)(?:[:=;8]['-]?[)d]+|[)d]+['-]?[:=;8])"), ' smile_token '),
    (re.compile(r"(?i)[:=;8]['-]?p+"), ' laugh_token '),
    (re.compile(r"(?i)(?:[:=;8]['-]?\(+|\)+['-]?[:=;8])"), ' sadface_token '),
    (re.compile(r"(?i)[:=;8]['-]?(?:[/|l*])"), ' neutralface_token '),
)


### Preprocessing Utils

In [ ]:
@dataclass(frozen=True)
class TextPreprocessingConfig:
    """Versioned and fully explicit policy for tweet text preprocessing."""

    profile: str
    normalize_whitespace: bool = True
    decode_html_entities: bool = False
    expand_contractions: bool = False
    replace_urls: bool = False
    replace_emails: bool = False
    replace_mentions: bool = False
    expand_hashtags: bool = False
    demojize_emojis: bool = False
    replace_ascii_emoticons: bool = False
    reduce_elongated_tokens: bool = False
    replace_numbers: bool = False
    encode_sentiment_punctuation: bool = False
    remove_digits: bool = False
    remove_punctuation: bool = False

    def __post_init__(self) -> None:
        """Reject mutually exclusive preprocessing options at configuration time."""
        if self.replace_numbers and self.remove_digits:
            raise ValueError('replace_numbers and remove_digits cannot both be enabled.')
        if self.encode_sentiment_punctuation and self.remove_punctuation:
            raise ValueError(
                'encode_sentiment_punctuation and remove_punctuation cannot both be enabled.'
            )

    @classmethod
    def whitespace_v1(cls) -> 'TextPreprocessingConfig':
        """Return the frozen minimal preprocessing baseline."""
        return cls(profile='whitespace_v1')

    @classmethod
    def social_text_v1(cls, **parameters: bool) -> 'TextPreprocessingConfig':
        """Return configurable social-text preprocessing with signal-preserving defaults."""
        defaults = SOCIAL_TEXT_V1_DEFAULT_PARAMETERS
        unexpected = sorted(set(parameters).difference(defaults))
        if unexpected:
            raise ValueError(f'Unsupported social_text_v1 parameter(s): {unexpected}')
        return cls(profile='social_text_v1', **{**defaults, **parameters})

    def as_mlflow_params(self) -> dict[str, str | int | float]:
        """Flatten the complete preprocessing policy for MLflow."""
        return {
            f'preprocessing_{name}': str(value).lower() if isinstance(value, bool) else value
            for name, value in asdict(self).items()
        }



def preprocess_one_text(text: str, config: TextPreprocessingConfig) -> str:
    """Apply an ordered, configurable social-text transformation to one tweet."""
    if config.decode_html_entities:
        text = html.unescape(text)
    if config.replace_emails:
        text = _EMAIL_PATTERN.sub(' email_token ', text)
    if config.replace_urls:
        text = _URL_PATTERN.sub(' url_token ', text)
    if config.replace_mentions:
        text = _MENTION_PATTERN.sub(' user_token ', text)
    if config.expand_hashtags:
        text = _HASHTAG_PATTERN.sub(r' hashtag_token \1 ', text)
    if config.expand_contractions:
        text = fix(text)
    if config.demojize_emojis:
        text = emoji.demojize(text, delimiters=(' emoji_', ' '))
    if config.replace_ascii_emoticons:
        for pattern, replacement in _ASCII_EMOTICON_REPLACEMENTS:
            text = pattern.sub(replacement, text)
        text = text.replace('<3', ' heart_token ')
    if config.reduce_elongated_tokens:
        text = re.sub(r'([A-Za-z])\1{2,}', r'\1\1', text)
    if config.replace_numbers:
        text = _NUMBER_PATTERN.sub(' number_token ', text)
    elif config.remove_digits:
        text = re.sub(r'\d+', ' ', text)
    if config.encode_sentiment_punctuation:
        text = re.sub(r'!{2,}', ' repeated_exclamation_token ', text)
        text = re.sub(r'\?{2,}', ' repeated_question_token ', text)
        text = text.replace('!', ' exclamation_token ').replace('?', ' question_token ')
    elif config.remove_punctuation:
        text = _PUNCTUATION_PATTERN.sub(' ', text)
    if config.normalize_whitespace:
        text = re.sub(r'\s+', ' ', text)
    return text.strip()


def preprocess_texts(texts: Iterable[object], config: TextPreprocessingConfig) -> np.ndarray:
    """Coerce missing values and apply a preprocessing policy to every input text."""
    values = pd.Series(texts, copy=False).fillna('').astype(str)
    return values.map(lambda text: preprocess_one_text(text, config)).to_numpy()


def normalize_text(texts: Iterable[object]) -> np.ndarray:
    """Retain the minimal historical normalizer as the whitespace_v1 profile."""
    return preprocess_texts(texts, TextPreprocessingConfig.whitespace_v1())


class ConfigurableTextPreprocessor(BaseEstimator, TransformerMixin):
    """Scikit-learn transformer that makes the selected preprocessing serializable."""

    def __init__(self, config: TextPreprocessingConfig):
        """Store the versioned policy used when scikit-learn transforms text."""
        self.config = config

    def fit(self, X, y=None):
        """Return the stateless transformer unchanged to satisfy scikit-learn."""
        return self

    def transform(self, X) -> np.ndarray:
        """Apply the configured text policy to each input tweet."""
        return preprocess_texts(X, self.config)



### Model Training Utils

In [ ]:
class CandidateSelectionError(ValueError):
    """Raised when no candidate satisfies the benchmark selection rule."""


@dataclass(frozen=True)
class TfidfLogisticConfig:
    """One reproducible TF-IDF and logistic-regression configuration."""

    name: str
    preprocessing: TextPreprocessingConfig
    ngram_range: tuple[int, int]
    min_df: int
    max_df: float
    max_features: int
    c: float
    penalty: str = 'l2'
    sublinear_tf: bool = True
    tol: float = 1e-4
    max_iter: int = 1000

    def as_mlflow_params(self) -> dict[str, str | int | float]:
        """Flatten the TF-IDF and logistic-regression settings for MLflow."""
        return {
            **self.preprocessing.as_mlflow_params(),
            'embedding_type': 'tfidf_word',
            'tfidf_ngram_range': f'{self.ngram_range[0]},{self.ngram_range[1]}',
            'tfidf_min_df': self.min_df,
            'tfidf_max_df': self.max_df,
            'tfidf_max_features': self.max_features,
            'tfidf_lowercase': 'true',
            'tfidf_strip_accents': 'unicode',
            'tfidf_sublinear_tf': str(self.sublinear_tf).lower(),
            'classifier_type': 'logistic_regression',
            'classifier_solver': 'liblinear',
            'classifier_penalty': self.penalty,
            'classifier_c': self.c,
            'classifier_tol': self.tol,
            'classifier_max_iter': self.max_iter,
            'classifier_random_state': 'PROJECT_SEED',
        }


@dataclass
class CandidateResult:
    """Fitted candidate and its validation outputs."""

    config: TfidfLogisticConfig
    pipeline: Pipeline
    validation_probabilities: np.ndarray
    validation_metrics: dict[str, float]
    train_time_seconds: float
    validation_inference_time_seconds: float

    def summary_record(self) -> dict[str, str | int | float]:
        """Combine one candidate's configuration, validation metrics and timings."""
        return {
            'configuration_name': self.config.name,
            **self.config.as_mlflow_params(),
            **self.validation_metrics,
            'training_time_seconds': self.train_time_seconds,
            'validation_inference_time_seconds': self.validation_inference_time_seconds,
            'validation_inference_ms_per_tweet': self.validation_inference_time_seconds * 1000 / len(self.validation_probabilities),
        }


def default_baseline_configuration(preprocessing: TextPreprocessingConfig) -> TfidfLogisticConfig:
    """Return the single fixed reference configuration for this model family."""
    return TfidfLogisticConfig(
        name='baseline_unigram_c1',
        preprocessing=preprocessing,
        ngram_range=(1, 1),
        min_df=2,
        max_df=1.0,
        max_features=75_000,
        c=1.0,
    )


def suggest_optuna_configuration(
    trial: optuna.trial.Trial,
    preprocessing_profiles: dict[str, TextPreprocessingConfig],
) -> TfidfLogisticConfig:
    """Sample one compact, valid TF-IDF plus liblinear configuration."""
    preprocessing_profile = trial.suggest_categorical(
        'preprocessing_profile', list(preprocessing_profiles)
    )
    return TfidfLogisticConfig(
        name=f'optuna_trial_{trial.number:03d}',
        preprocessing=preprocessing_profiles[preprocessing_profile],
        ngram_range=(1, trial.suggest_categorical('tfidf_ngram_max', [1, 2])),
        min_df=trial.suggest_categorical('tfidf_min_df', [2, 5, 10]),
        max_df=trial.suggest_categorical('tfidf_max_df', [0.95, 1.0]),
        max_features=trial.suggest_categorical('tfidf_max_features', [75_000, 120_000]),
        c=trial.suggest_float('classifier_c', 0.1, 5.0, log=True),
        penalty=trial.suggest_categorical('classifier_penalty', ['l1', 'l2']),
    )


def build_pipeline(config: TfidfLogisticConfig, random_state: int) -> Pipeline:
    """Build the explicit preprocessing, TF-IDF and Logistic Regression pipeline."""
    return Pipeline(steps=[
        ('preprocess', ConfigurableTextPreprocessor(config.preprocessing)),
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            sublinear_tf=config.sublinear_tf,
            ngram_range=config.ngram_range,
            min_df=config.min_df,
            max_df=config.max_df,
            max_features=config.max_features,
        )),
        ('classifier', LogisticRegression(
            C=config.c,
            solver='liblinear',
            penalty=config.penalty,
            tol=config.tol,
            max_iter=config.max_iter,
            random_state=random_state,
        )),
    ])


def probabilities_for_positive_class(pipeline: Pipeline, texts: Iterable[object]) -> np.ndarray:
    """Return P(y=1) without assuming the order of fitted classifier classes."""
    probabilities = pipeline.predict_proba(texts)
    classes = pipeline.named_steps['classifier'].classes_
    positive_index = int(np.where(classes == POSITIVE_LABEL)[0][0])
    return probabilities[:, positive_index]


def labels_from_probabilities(probabilities: np.ndarray, threshold: float) -> np.ndarray:
    """Return label 1 when its probability meets the frozen threshold."""
    if not 0.0 < threshold < 1.0:
        raise ValueError('threshold must be strictly between 0 and 1.')
    return (np.asarray(probabilities) >= threshold).astype(int)


def evaluate_binary_predictions(y_true: Iterable[int], y_pred: Iterable[int]) -> dict[str, float]:
    """Compute shared metrics for the negative (bad-buzz) class."""
    truth, predictions = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)
    negative_count = int((truth == NEGATIVE_LABEL).sum())
    missed_negative_tweets = (truth == NEGATIVE_LABEL) & (predictions != NEGATIVE_LABEL)
    badbuzz_miss_rate = np.nan if negative_count == 0 else BADBUZZ_REFERENCE_VOLUME * missed_negative_tweets.sum() / negative_count
    return {
        'accuracy': float(accuracy_score(truth, predictions)),
        'precision_negative': float(precision_score(truth, predictions, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0)),
        'recall_negative': float(recall_score(truth, predictions, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0)),
        'fbeta_negative': float(fbeta_score(truth, predictions, beta=FBETA_BETA, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0)),
        'badbuzz_miss_per_1000_negative_tweets': float(badbuzz_miss_rate),
    }


def fit_and_evaluate_candidate(config, train_texts, train_labels, validation_texts, validation_labels, random_state):
    """Fit one configuration and evaluate its default 0.5 threshold on validation."""
    pipeline = build_pipeline(config, random_state=random_state)
    train_start = perf_counter()
    pipeline.fit(train_texts, np.asarray(train_labels, dtype=int))
    train_time_seconds = perf_counter() - train_start
    inference_start = perf_counter()
    validation_probabilities = probabilities_for_positive_class(pipeline, validation_texts)
    validation_inference_time_seconds = perf_counter() - inference_start
    validation_predictions = labels_from_probabilities(validation_probabilities, threshold=0.5)
    return CandidateResult(
        config=config, pipeline=pipeline, validation_probabilities=validation_probabilities,
        validation_metrics=evaluate_binary_predictions(validation_labels, validation_predictions),
        train_time_seconds=train_time_seconds, validation_inference_time_seconds=validation_inference_time_seconds,
    )


def select_by_validation_rule(candidates: pd.DataFrame) -> pd.Series:
    """Apply accuracy eligibility, F-beta ranking and documented tie-breakers."""
    eligible = candidates.loc[candidates['accuracy'] >= MINIMUM_VALIDATION_ACCURACY].copy()
    if eligible.empty:
        raise CandidateSelectionError(f'No candidate reached the minimum validation accuracy of {MINIMUM_VALIDATION_ACCURACY:.0%}.')
    return eligible.sort_values(
        by=['fbeta_negative', 'badbuzz_miss_per_1000_negative_tweets', 'accuracy'],
        ascending=[False, True, False], kind='mergesort',
    ).iloc[0]


def search_thresholds(y_true, positive_probabilities, thresholds=None, default_threshold=0.5):
    """Evaluate a bounded validation grid while retaining 0.5 explicitly."""
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 181)
    grid = np.unique(np.round(np.append(np.asarray(list(thresholds), dtype=float), default_threshold), decimals=12))
    records = []
    for threshold in grid:
        records.append({
            'threshold': float(threshold),
            'is_default_threshold': bool(np.isclose(threshold, default_threshold)),
            **evaluate_binary_predictions(y_true, labels_from_probabilities(positive_probabilities, float(threshold))),
        })
    return pd.DataFrame.from_records(records).sort_values('threshold').reset_index(drop=True)


def log_dataframe_artifact(dataframe, filename):
    """Log a CSV artifact without persisting temporary files in the repository."""
    with tempfile.TemporaryDirectory(prefix='benchmark-artifact-') as temporary_directory:
        artifact_path = Path(temporary_directory) / filename
        dataframe.to_csv(artifact_path, index=False)
        mlflow.log_artifact(str(artifact_path))


def log_json_artifact(payload, filename):
    """Log a JSON artifact without exposing source data or credentials."""
    with tempfile.TemporaryDirectory(prefix='benchmark-artifact-') as temporary_directory:
        artifact_path = Path(temporary_directory) / filename
        artifact_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        mlflow.log_artifact(str(artifact_path))


In [ ]:
# Generic neural-model helpers, used by GloVe + LSTM and reusable by later models.
GLOVE_TWITTER_27B_FILENAME = 'glove.twitter.27B.50d.txt'
GLOVE_TWITTER_27B_DIMENSION = 50
TOKENIZER_FILTERS_PRESERVING_UNDERSCORE = Tokenizer().filters.replace('_', '')


@dataclass(frozen=True)
class GloveLstmConfig:
    """One reproducible GloVe plus LSTM configuration."""

    name: str
    preprocessing: TextPreprocessingConfig
    vocabulary_size: int
    max_sequence_length: int
    lstm_units: int
    dropout: float
    learning_rate: float
    batch_size: int
    max_epochs: int
    early_stopping_patience: int
    embedding_dimension: int = GLOVE_TWITTER_27B_DIMENSION
    recurrent_dropout: float = 0.0
    embedding_trainable: bool = False

    def as_mlflow_params(self) -> dict[str, str | int | float]:
        """Flatten the GloVe, tokenizer, LSTM and training settings for MLflow."""
        return {
            **self.preprocessing.as_mlflow_params(),
            'embedding_type': 'glove_twitter_27b',
            'embedding_dimension': self.embedding_dimension,
            'embedding_trainable': str(self.embedding_trainable).lower(),
            'tokenizer_type': 'tensorflow_keras_tokenizer',
            'tokenizer_lowercase': 'true',
            'tokenizer_oov_token': 'oov_token',
            'tokenizer_filters_preserve_underscore': 'true',
            'tokenizer_vocabulary_size_limit': self.vocabulary_size,
            'sequence_max_length': self.max_sequence_length,
            'sequence_padding': 'post',
            'sequence_truncating': 'post',
            'classifier_type': 'lstm',
            'lstm_units': self.lstm_units,
            'lstm_dropout': self.dropout,
            'lstm_recurrent_dropout': self.recurrent_dropout,
            'optimizer_type': 'adam',
            'optimizer_learning_rate': self.learning_rate,
            'loss': 'binary_crossentropy',
            'batch_size': self.batch_size,
            'max_epochs': self.max_epochs,
            'early_stopping_monitor': 'val_loss',
            'early_stopping_patience': self.early_stopping_patience,
            'candidate_seed': 'PROJECT_SEED',
        }


@dataclass
class GloveLstmCandidateResult:
    """Fitted neural candidate and validation-only outputs."""

    config: GloveLstmConfig
    model: keras.Model
    validation_probabilities: np.ndarray
    validation_metrics: dict[str, float]
    history: dict[str, list[float]]
    coverage: dict[str, float]
    train_time_seconds: float
    input_preparation_time_seconds: float
    validation_inference_time_seconds: float
    effective_vocabulary_size: int
    best_epoch: int

    def summary_record(self) -> dict[str, str | int | float]:
        """Combine neural configuration, coverage, validation metrics and timings."""
        return {
            'configuration_name': self.config.name,
            **self.config.as_mlflow_params(),
            **self.coverage,
            **self.validation_metrics,
            'effective_vocabulary_size': self.effective_vocabulary_size,
            'training_epoch_count': len(self.history.get('loss', [])),
            'best_epoch': self.best_epoch,
            'input_preparation_time_seconds': self.input_preparation_time_seconds,
            'training_time_seconds': self.train_time_seconds,
            'validation_inference_time_seconds': self.validation_inference_time_seconds,
            'validation_inference_ms_per_tweet': (
                self.validation_inference_time_seconds * 1000 / len(self.validation_probabilities)
            ),
        }


def resolve_glove_embedding_path(path_override=None, filename=GLOVE_TWITTER_27B_FILENAME) -> Path:
    """Resolve a local or attached Kaggle GloVe asset without downloading it."""
    if path_override:
        candidate = Path(path_override).expanduser()
        if candidate.is_file():
            return candidate.resolve()
        raise FileNotFoundError(f'GLOVE_EMBEDDING_PATH does not point to a file: {candidate}')

    candidates = [
        Path.cwd() / 'data' / filename,
        Path.cwd() / filename,
        Path('/kaggle/working') / filename,
    ]
    if _is_kaggle_runtime():
        input_root = Path('/kaggle/input')
        candidates.extend(
            folder / filename
            for folder in [input_root, *input_root.rglob('*')]
            if folder.is_dir()
        )

    matches = []
    for candidate in candidates:
        if candidate.is_file() and candidate.resolve() not in matches:
            matches.append(candidate.resolve())
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        locations = '\n'.join(f'- {path}' for path in matches)
        raise FileNotFoundError(
            f'Multiple {filename} files were found. Set GLOVE_EMBEDDING_PATH explicitly:\n{locations}'
        )
    placement = (
        'Attach a Kaggle dataset containing the file.'
        if _is_kaggle_runtime()
        else 'Place the file in data/ or set GLOVE_EMBEDDING_PATH.'
    )
    raise FileNotFoundError(f'Could not find {filename}. {placement}')


def build_glove_manifest(asset_path: Path, expected_dimension: int) -> dict[str, str | int]:
    """Validate the vector dimension and create a traceable asset manifest."""
    with asset_path.open('r', encoding='utf-8') as handle:
        first_vector = next((line.rstrip().split() for line in handle if line.strip()), None)
    if not first_vector:
        raise ValueError(f'GloVe embedding file is empty: {asset_path}')
    if len(first_vector) - 1 != expected_dimension:
        raise ValueError(
            f'Expected {expected_dimension} dimensions in {asset_path.name}, '
            f'found {len(first_vector) - 1}.'
        )
    return {
        'source_path': str(asset_path),
        'filename': asset_path.name,
        'byte_size': asset_path.stat().st_size,
        'embedding_dimension': expected_dimension,
    }


def effective_vocabulary_size(tokenizer: Tokenizer, requested_size: int) -> int:
    """Return the Keras embedding input dimension for a capped tokenizer."""
    if requested_size < 2:
        raise ValueError('vocabulary_size must be at least 2.')
    return min(requested_size, len(tokenizer.word_index) + 1)


def prepare_glove_lstm_resources(
    train_texts,
    validation_texts,
    preprocessing: TextPreprocessingConfig,
    asset_path: Path,
    maximum_vocabulary_size: int,
    embedding_dimension: int,
) -> dict:
    """Preprocess train/validation only and load relevant GloVe vectors once."""
    train_clean = preprocess_texts(train_texts, preprocessing)
    validation_clean = preprocess_texts(validation_texts, preprocessing)
    tokenizer = Tokenizer(
        num_words=maximum_vocabulary_size,
        filters=TOKENIZER_FILTERS_PRESERVING_UNDERSCORE,
        lower=True,
        split=' ',
        oov_token='oov_token',
    )
    tokenizer.fit_on_texts(train_clean)
    maximum_size = effective_vocabulary_size(tokenizer, maximum_vocabulary_size)
    wanted_words = {
        word for word, index in tokenizer.word_index.items()
        if 0 < index < maximum_size and word != tokenizer.oov_token
    }

    vectors = {}
    with asset_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            values = line.rstrip().split()
            if not values or values[0] not in wanted_words:
                continue
            if len(values) != embedding_dimension + 1:
                raise ValueError(f'Malformed GloVe vector for token {values[0]!r}.')
            vectors[values[0]] = np.asarray(values[1:], dtype=np.float32)

    return {
        'manifest': build_glove_manifest(asset_path, embedding_dimension),
        'tokenizer': tokenizer,
        'train_clean': train_clean,
        'validation_clean': validation_clean,
        'vectors': vectors,
        'maximum_vocabulary_size': maximum_vocabulary_size,
    }


def texts_to_padded_sequences(
    tokenizer: Tokenizer,
    texts,
    vocabulary_size: int,
    max_sequence_length: int,
) -> np.ndarray:
    """Apply one vocabulary cap without refitting the train-only tokenizer."""
    original_num_words = tokenizer.num_words
    tokenizer.num_words = effective_vocabulary_size(tokenizer, vocabulary_size)
    try:
        sequences = tokenizer.texts_to_sequences(texts)
    finally:
        tokenizer.num_words = original_num_words
    return pad_sequences(
        sequences,
        maxlen=max_sequence_length,
        padding='post',
        truncating='post',
        dtype='int32',
        value=0,
    )


def build_glove_embedding_matrix(
    tokenizer: Tokenizer,
    vectors: dict[str, np.ndarray],
    vocabulary_size: int,
    embedding_dimension: int,
) -> tuple[np.ndarray, dict[str, float]]:
    """Create one zero-padded embedding matrix and its train-vocabulary coverage."""
    effective_size = effective_vocabulary_size(tokenizer, vocabulary_size)
    matrix = np.zeros((effective_size, embedding_dimension), dtype=np.float32)
    vocabulary = [
        (word, index) for word, index in tokenizer.word_index.items()
        if 0 < index < effective_size and word != tokenizer.oov_token
    ]
    matched = 0
    for word, index in vocabulary:
        if word in vectors:
            matrix[index] = vectors[word]
            matched += 1
    vocabulary_count = len(vocabulary)
    return matrix, {
        'glove_vocabulary_token_count': float(vocabulary_count),
        'glove_matched_token_count': float(matched),
        'glove_unmatched_token_count': float(vocabulary_count - matched),
        'glove_coverage_ratio': float(matched / vocabulary_count) if vocabulary_count else 0.0,
    }


def build_glove_lstm_model(config: GloveLstmConfig, embedding_matrix: np.ndarray) -> keras.Model:
    """Build the historical frozen-GloVe LSTM architecture."""
    if embedding_matrix.ndim != 2 or embedding_matrix.shape[1] != config.embedding_dimension:
        raise ValueError('Embedding matrix dimensions are incompatible with the configuration.')
    model = keras.Sequential([
        keras.layers.Input(shape=(config.max_sequence_length,), dtype='int32'),
        keras.layers.Embedding(
            input_dim=embedding_matrix.shape[0],
            output_dim=config.embedding_dimension,
            embeddings_initializer=keras.initializers.Constant(embedding_matrix),
            mask_zero=True,
            trainable=config.embedding_trainable,
            name='glove_embedding',
        ),
        keras.layers.LSTM(
            config.lstm_units,
            dropout=config.dropout,
            recurrent_dropout=config.recurrent_dropout,
            name='lstm',
        ),
        keras.layers.Dense(1, activation='sigmoid', name='sentiment_probability'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


def reset_tensorflow_candidate_state(seed: int) -> None:
    """Free stale model state and reset the candidate seed."""
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)


def fit_and_evaluate_glove_lstm_candidate(
    config: GloveLstmConfig,
    resources: dict,
    train_labels,
    validation_labels,
    random_state: int,
) -> GloveLstmCandidateResult:
    """Fit one candidate using train/validation only, at the default 0.5 threshold."""
    if config.embedding_dimension != resources['manifest']['embedding_dimension']:
        raise ValueError('The GloVe asset dimension does not match the candidate.')

    reset_tensorflow_candidate_state(random_state)
    preparation_start = perf_counter()
    matrix, coverage = build_glove_embedding_matrix(
        resources['tokenizer'], resources['vectors'], config.vocabulary_size, config.embedding_dimension
    )
    train_sequences = texts_to_padded_sequences(
        resources['tokenizer'], resources['train_clean'], config.vocabulary_size, config.max_sequence_length
    )
    validation_sequences = texts_to_padded_sequences(
        resources['tokenizer'], resources['validation_clean'], config.vocabulary_size, config.max_sequence_length
    )
    model = build_glove_lstm_model(config, matrix)
    input_preparation_time_seconds = perf_counter() - preparation_start

    history_object = model.fit(
        train_sequences,
        np.asarray(train_labels, dtype=np.int32),
        validation_data=(validation_sequences, np.asarray(validation_labels, dtype=np.int32)),
        batch_size=config.batch_size,
        epochs=config.max_epochs,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=config.early_stopping_patience,
            mode='min',
            restore_best_weights=True,
            verbose=1,
        )],
        verbose=2,
        shuffle=True,
    )
    train_time_seconds = perf_counter() - preparation_start - input_preparation_time_seconds
    history = {name: [float(value) for value in values] for name, values in history_object.history.items()}

    inference_start = perf_counter()
    validation_probabilities = model.predict(
        validation_sequences, batch_size=config.batch_size, verbose=0
    ).reshape(-1)
    validation_inference_time_seconds = perf_counter() - inference_start
    validation_predictions = labels_from_probabilities(validation_probabilities, threshold=0.5)
    validation_losses = history.get('val_loss', [])
    best_epoch = int(np.argmin(validation_losses) + 1) if validation_losses else len(history.get('loss', []))

    return GloveLstmCandidateResult(
        config=config,
        model=model,
        validation_probabilities=validation_probabilities,
        validation_metrics=evaluate_binary_predictions(validation_labels, validation_predictions),
        history=history,
        coverage=coverage,
        train_time_seconds=train_time_seconds,
        input_preparation_time_seconds=input_preparation_time_seconds,
        validation_inference_time_seconds=validation_inference_time_seconds,
        effective_vocabulary_size=matrix.shape[0],
        best_epoch=best_epoch,
    )


def log_figure_artifact(figure, filename: str) -> None:
    """Log a Matplotlib figure through a temporary, model-neutral artifact path."""
    with tempfile.TemporaryDirectory(prefix='benchmark-artifact-') as temporary_directory:
        artifact_path = Path(temporary_directory) / filename
        figure.savefig(artifact_path, dpi=150, bbox_inches='tight')
        mlflow.log_artifact(str(artifact_path))


def log_neural_learning_history(history: dict[str, list[float]]) -> pd.DataFrame:
    """Log generic neural training history and learning curves for the active run."""
    history_frame = pd.DataFrame(history)
    history_frame.index = history_frame.index + 1
    history_frame.index.name = 'epoch'
    log_dataframe_artifact(history_frame.reset_index(), 'training_history.csv')
    aliases = {'loss': 'training_loss', 'accuracy': 'training_accuracy',
               'val_loss': 'validation_loss', 'val_accuracy': 'validation_accuracy'}
    for epoch, row in history_frame.iterrows():
        mlflow.log_metrics(
            {f'epoch_{aliases.get(name, name)}': float(value)
             for name, value in row.items() if pd.notna(value)},
            step=int(epoch),
        )

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for axis, metric, validation_metric, title in [
        (axes[0], 'loss', 'val_loss', 'Loss'),
        (axes[1], 'accuracy', 'val_accuracy', 'Accuracy'),
    ]:
        if metric in history_frame:
            axis.plot(history_frame.index, history_frame[metric], label='train')
        if validation_metric in history_frame:
            axis.plot(history_frame.index, history_frame[validation_metric], label='validation')
        axis.set_title(title)
        axis.set_xlabel('Epoch')
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend()
    figure.tight_layout()
    log_figure_artifact(figure, 'learning_curves.png')
    plt.close(figure)
    return history_frame


def tokenizer_artifact_payload(tokenizer: Tokenizer, vocabulary_size: int) -> dict:
    """Serialize the selected tokenizer with its effective vocabulary cap."""
    original_num_words = tokenizer.num_words
    tokenizer.num_words = effective_vocabulary_size(tokenizer, vocabulary_size)
    try:
        return json.loads(tokenizer.to_json())
    finally:
        tokenizer.num_words = original_num_words


In [ ]:
# Word2Vec + GRU helpers. Word2Vec is trained once on train-only text and reused by every GRU candidate.


@dataclass(frozen=True)
class Word2VecTrainingConfig:
    """Fixed, reproducible Gensim Word2Vec settings for one benchmark run."""

    vector_size: int
    window: int
    min_count: int
    sg: int
    negative: int
    epochs: int
    seed: int
    workers: int = 1

    def as_mlflow_params(self) -> dict[str, str | int | float]:
        """Flatten the fixed train-only Word2Vec configuration for MLflow."""
        return {
            'embedding_type': 'word2vec_train_only',
            'word2vec_vector_size': self.vector_size,
            'word2vec_window': self.window,
            'word2vec_min_count': self.min_count,
            'word2vec_algorithm': 'skip_gram' if self.sg else 'cbow',
            'word2vec_negative_samples': self.negative,
            'word2vec_epochs': self.epochs,
            'word2vec_seed': self.seed,
            'word2vec_workers': self.workers,
        }


@dataclass(frozen=True)
class Word2VecGruConfig:
    """One reproducible Word2Vec-initialized GRU configuration."""

    name: str
    preprocessing: TextPreprocessingConfig
    vocabulary_size: int
    max_sequence_length: int
    gru_units: int
    dropout: float
    learning_rate: float
    batch_size: int
    max_epochs: int
    early_stopping_patience: int
    embedding_dimension: int
    recurrent_dropout: float = 0.0
    embedding_trainable: bool = False

    def as_mlflow_params(self) -> dict[str, str | int | float]:
        """Flatten tokenizer, embedding, GRU and training settings for MLflow."""
        return {
            **self.preprocessing.as_mlflow_params(),
            'embedding_type': 'word2vec_train_only',
            'embedding_dimension': self.embedding_dimension,
            'embedding_trainable': str(self.embedding_trainable).lower(),
            'tokenizer_type': 'tensorflow_keras_tokenizer',
            'tokenizer_lowercase': 'true',
            'tokenizer_oov_token': 'oov_token',
            'tokenizer_filters_preserve_underscore': 'true',
            'tokenizer_vocabulary_size_limit': self.vocabulary_size,
            'sequence_max_length': self.max_sequence_length,
            'sequence_padding': 'post',
            'sequence_truncating': 'post',
            'classifier_type': 'gru',
            'gru_units': self.gru_units,
            'gru_dropout': self.dropout,
            'gru_recurrent_dropout': self.recurrent_dropout,
            'optimizer_type': 'adam',
            'optimizer_learning_rate': self.learning_rate,
            'loss': 'binary_crossentropy',
            'batch_size': self.batch_size,
            'max_epochs': self.max_epochs,
            'early_stopping_monitor': 'val_loss',
            'early_stopping_patience': self.early_stopping_patience,
            'candidate_seed': 'PROJECT_SEED',
        }


@dataclass
class Word2VecGruCandidateResult:
    """Fitted Word2Vec/GRU candidate and its validation-only outputs."""

    config: Word2VecGruConfig
    model: keras.Model
    validation_probabilities: np.ndarray
    validation_metrics: dict[str, float]
    history: dict[str, list[float]]
    coverage: dict[str, float]
    train_time_seconds: float
    input_preparation_time_seconds: float
    validation_inference_time_seconds: float
    effective_vocabulary_size: int
    best_epoch: int

    def summary_record(self) -> dict[str, str | int | float]:
        """Combine GRU configuration, Word2Vec coverage, validation metrics and timings."""
        return {
            'configuration_name': self.config.name,
            **self.config.as_mlflow_params(),
            **self.coverage,
            **self.validation_metrics,
            'effective_vocabulary_size': self.effective_vocabulary_size,
            'training_epoch_count': len(self.history.get('loss', [])),
            'best_epoch': self.best_epoch,
            'input_preparation_time_seconds': self.input_preparation_time_seconds,
            'training_time_seconds': self.train_time_seconds,
            'validation_inference_time_seconds': self.validation_inference_time_seconds,
            'validation_inference_ms_per_tweet': (
                self.validation_inference_time_seconds * 1000 / len(self.validation_probabilities)
            ),
        }


def prepare_word2vec_gru_resources(
    train_texts,
    validation_texts,
    preprocessing: TextPreprocessingConfig,
    word2vec_config: Word2VecTrainingConfig,
    maximum_vocabulary_size: int,
) -> dict:
    """Fit Word2Vec and the tokenizer on train-only text, then prepare validation text."""
    train_clean = preprocess_texts(train_texts, preprocessing)
    validation_clean = preprocess_texts(validation_texts, preprocessing)
    sentences = [text.lower().split() for text in train_clean if text.strip()]
    if not sentences:
        raise ValueError('Word2Vec requires at least one non-empty preprocessed training tweet.')

    training_start = perf_counter()
    word2vec_model = Word2Vec(
        sentences=sentences,
        vector_size=word2vec_config.vector_size,
        window=word2vec_config.window,
        min_count=word2vec_config.min_count,
        sg=word2vec_config.sg,
        negative=word2vec_config.negative,
        epochs=word2vec_config.epochs,
        seed=word2vec_config.seed,
        workers=word2vec_config.workers,
    )
    word2vec_training_time_seconds = perf_counter() - training_start

    tokenizer = Tokenizer(
        num_words=maximum_vocabulary_size,
        filters=TOKENIZER_FILTERS_PRESERVING_UNDERSCORE,
        lower=True,
        split=' ',
        oov_token='oov_token',
    )
    tokenizer.fit_on_texts(train_clean)
    return {
        'word2vec_model': word2vec_model,
        'word2vec_config': word2vec_config,
        'tokenizer': tokenizer,
        'train_clean': train_clean,
        'validation_clean': validation_clean,
        'maximum_vocabulary_size': maximum_vocabulary_size,
        'manifest': {
            'embedding_source': 'word2vec_trained_on_train_only',
            'embedding_dimension': word2vec_config.vector_size,
            'word2vec_sentence_count': len(sentences),
            'word2vec_vocabulary_count': len(word2vec_model.wv),
            'word2vec_training_time_seconds': word2vec_training_time_seconds,
        },
    }


def build_word2vec_embedding_matrix(
    tokenizer: Tokenizer,
    keyed_vectors,
    vocabulary_size: int,
    embedding_dimension: int,
) -> tuple[np.ndarray, dict[str, float]]:
    """Create a zero-padded matrix from train-only Word2Vec vectors and report coverage."""
    effective_size = effective_vocabulary_size(tokenizer, vocabulary_size)
    matrix = np.zeros((effective_size, embedding_dimension), dtype=np.float32)
    vocabulary = [
        (word, index) for word, index in tokenizer.word_index.items()
        if 0 < index < effective_size and word != tokenizer.oov_token
    ]
    matched = 0
    for word, index in vocabulary:
        if word in keyed_vectors:
            matrix[index] = keyed_vectors[word]
            matched += 1
    vocabulary_count = len(vocabulary)
    return matrix, {
        'word2vec_vocabulary_token_count': float(vocabulary_count),
        'word2vec_matched_token_count': float(matched),
        'word2vec_unmatched_token_count': float(vocabulary_count - matched),
        'word2vec_coverage_ratio': float(matched / vocabulary_count) if vocabulary_count else 0.0,
    }


def build_word2vec_gru_model(config: Word2VecGruConfig, embedding_matrix: np.ndarray) -> keras.Model:
    """Build the frozen-Word2Vec GRU architecture used by this benchmark family."""
    if embedding_matrix.ndim != 2 or embedding_matrix.shape[1] != config.embedding_dimension:
        raise ValueError('Embedding matrix dimensions are incompatible with the configuration.')
    model = keras.Sequential([
        keras.layers.Input(shape=(config.max_sequence_length,), dtype='int32'),
        keras.layers.Embedding(
            input_dim=embedding_matrix.shape[0],
            output_dim=config.embedding_dimension,
            embeddings_initializer=keras.initializers.Constant(embedding_matrix),
            mask_zero=True,
            trainable=config.embedding_trainable,
            name='word2vec_embedding',
        ),
        keras.layers.GRU(
            config.gru_units,
            dropout=config.dropout,
            recurrent_dropout=config.recurrent_dropout,
            name='gru',
        ),
        keras.layers.Dense(1, activation='sigmoid', name='sentiment_probability'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model


def fit_and_evaluate_word2vec_gru_candidate(
    config: Word2VecGruConfig,
    resources: dict,
    train_labels,
    validation_labels,
    random_state: int,
) -> Word2VecGruCandidateResult:
    """Fit one GRU candidate with fixed train-only Word2Vec vectors and validation data."""
    if config.embedding_dimension != resources['manifest']['embedding_dimension']:
        raise ValueError('The Word2Vec dimension does not match the candidate.')

    reset_tensorflow_candidate_state(random_state)
    preparation_start = perf_counter()
    matrix, coverage = build_word2vec_embedding_matrix(
        resources['tokenizer'],
        resources['word2vec_model'].wv,
        config.vocabulary_size,
        config.embedding_dimension,
    )
    train_sequences = texts_to_padded_sequences(
        resources['tokenizer'],
        resources['train_clean'],
        config.vocabulary_size,
        config.max_sequence_length,
    )
    validation_sequences = texts_to_padded_sequences(
        resources['tokenizer'],
        resources['validation_clean'],
        config.vocabulary_size,
        config.max_sequence_length,
    )
    model = build_word2vec_gru_model(config, matrix)
    input_preparation_time_seconds = perf_counter() - preparation_start

    history_object = model.fit(
        train_sequences,
        np.asarray(train_labels, dtype=np.int32),
        validation_data=(validation_sequences, np.asarray(validation_labels, dtype=np.int32)),
        batch_size=config.batch_size,
        epochs=config.max_epochs,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=config.early_stopping_patience,
            mode='min',
            restore_best_weights=True,
            verbose=1,
        )],
        verbose=2,
        shuffle=True,
    )
    train_time_seconds = perf_counter() - preparation_start - input_preparation_time_seconds
    history = {name: [float(value) for value in values] for name, values in history_object.history.items()}

    inference_start = perf_counter()
    validation_probabilities = model.predict(
        validation_sequences,
        batch_size=config.batch_size,
        verbose=0,
    ).reshape(-1)
    validation_inference_time_seconds = perf_counter() - inference_start
    validation_predictions = labels_from_probabilities(validation_probabilities, threshold=DEFAULT_DECISION_THRESHOLD)
    validation_losses = history.get('val_loss', [])
    best_epoch = int(np.argmin(validation_losses) + 1) if validation_losses else len(history.get('loss', []))

    return Word2VecGruCandidateResult(
        config=config,
        model=model,
        validation_probabilities=validation_probabilities,
        validation_metrics=evaluate_binary_predictions(validation_labels, validation_predictions),
        history=history,
        coverage=coverage,
        train_time_seconds=train_time_seconds,
        input_preparation_time_seconds=input_preparation_time_seconds,
        validation_inference_time_seconds=validation_inference_time_seconds,
        effective_vocabulary_size=matrix.shape[0],
        best_epoch=best_epoch,
    )


def log_word2vec_model_artifact(word2vec_model: Word2Vec, filename: str = 'word2vec.model') -> None:
    """Log the train-only Gensim Word2Vec model without persisting it in the repository."""
    with tempfile.TemporaryDirectory(prefix='benchmark-artifact-') as temporary_directory:
        artifact_path = Path(temporary_directory) / filename
        word2vec_model.save(str(artifact_path), separately=[])
        mlflow.log_artifact(str(artifact_path))


In [ ]:
# Frozen DistilBERT classifier helpers. The backbone is downloaded from Hugging Face and never trained.


@dataclass(frozen=True)
class DistilBertClassifierConfig:
    """One reproducible frozen-DistilBERT and dense-head configuration."""

    name: str
    model_id: str
    model_revision: str
    preprocessing: TextPreprocessingConfig
    max_sequence_length: int
    classifier_hidden_units: int
    dropout: float
    learning_rate: float
    batch_size: int
    max_epochs: int
    early_stopping_patience: int
    backbone_trainable: bool = False

    def as_mlflow_params(self) -> dict[str, str | int | float]:
        """Flatten tokenizer, backbone, head and training settings for MLflow."""
        return {
            **self.preprocessing.as_mlflow_params(),
            'embedding_type': 'pretrained_transformer_encoder',
            'transformer_model_id': self.model_id,
            'transformer_model_revision': self.model_revision,
            'transformer_backbone_trainable': str(self.backbone_trainable).lower(),
            'tokenizer_type': 'huggingface_auto_tokenizer',
            'tokenizer_padding': 'max_length',
            'tokenizer_truncation': 'true',
            'sequence_max_length': self.max_sequence_length,
            'classifier_type': 'frozen_distilbert_dense_head',
            'classifier_hidden_units': self.classifier_hidden_units,
            'classifier_dropout': self.dropout,
            'optimizer_type': 'adamw',
            'optimizer_learning_rate': self.learning_rate,
            'loss': 'cross_entropy',
            'batch_size': self.batch_size,
            'max_epochs': self.max_epochs,
            'early_stopping_monitor': 'val_loss',
            'early_stopping_patience': self.early_stopping_patience,
            'candidate_seed': 'PROJECT_SEED',
        }


@dataclass
class DistilBertCandidateResult:
    """Fitted frozen-DistilBERT candidate and its validation-only outputs."""

    config: DistilBertClassifierConfig
    model: torch.nn.Module
    validation_probabilities: np.ndarray
    validation_metrics: dict[str, float]
    history: dict[str, list[float]]
    train_time_seconds: float
    input_preparation_time_seconds: float
    validation_inference_time_seconds: float
    best_epoch: int
    trainable_parameter_count: int
    total_parameter_count: int

    def summary_record(self) -> dict[str, str | int | float]:
        """Combine configuration, validation metrics, parameter counts and timings."""
        return {
            'configuration_name': self.config.name,
            **self.config.as_mlflow_params(),
            **self.validation_metrics,
            'training_epoch_count': len(self.history.get('loss', [])),
            'best_epoch': self.best_epoch,
            'trainable_parameter_count': self.trainable_parameter_count,
            'total_parameter_count': self.total_parameter_count,
            'input_preparation_time_seconds': self.input_preparation_time_seconds,
            'training_time_seconds': self.train_time_seconds,
            'validation_inference_time_seconds': self.validation_inference_time_seconds,
            'validation_inference_ms_per_tweet': self.validation_inference_time_seconds * 1000 / len(self.validation_probabilities),
        }


class TransformerTweetDataset(torch.utils.data.Dataset):
    """Keep preprocessed tweets as text until a DataLoader tokenizes a full batch."""

    def __init__(self, texts, labels):
        """Store aligned texts and binary labels without fitting on validation or test."""
        self.texts = [str(text) for text in texts]
        self.labels = np.asarray(labels, dtype=np.int64)
        if len(self.texts) != len(self.labels):
            raise ValueError('Transformer texts and labels must have identical lengths.')

    def __len__(self) -> int:
        """Return the number of tweets available to this split."""
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, str | int]:
        """Return one raw tweet and its label for batched tokenization."""
        return {'text': self.texts[index], 'target': int(self.labels[index])}


class FrozenTransformerClassifier(torch.nn.Module):
    """Apply masked mean pooling to a frozen encoder, then classify with a dense head."""

    def __init__(self, encoder, hidden_size: int, classifier_hidden_units: int, dropout: float):
        """Attach an immutable Transformer encoder to a trainable two-layer classifier."""
        super().__init__()
        self.encoder = encoder
        self.encoder.requires_grad_(False)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, classifier_hidden_units),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(classifier_hidden_units, 2),
        )

    def train(self, mode: bool = True):
        """Keep frozen encoder dropout disabled even while the dense head is trained."""
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, **encoder_inputs):
        """Return two-class logits from masked mean-pooled encoder representations."""
        with torch.no_grad():
            outputs = self.encoder(**encoder_inputs)
        token_embeddings = outputs.last_hidden_state
        attention_mask = encoder_inputs['attention_mask'].unsqueeze(-1).type_as(token_embeddings)
        sentence_embedding = (token_embeddings * attention_mask).sum(dim=1)
        sentence_embedding = sentence_embedding / attention_mask.sum(dim=1).clamp_min(1e-9)
        return self.classifier(sentence_embedding)


def _transformer_download_error(model_id: str, error: Exception) -> RuntimeError:
    """Build an actionable error when Hugging Face weights cannot be reached or cached."""
    return RuntimeError(
        f'Could not load {model_id!r} from Hugging Face. Enable Internet in Kaggle notebook settings '
        'for the first run, then restart the session if the download was interrupted.'
    )


def prepare_distilbert_resources(train_texts, validation_texts, preprocessing, model_id: str, model_revision: str) -> dict:
    """Download tokenizer/config once and prepare only train and validation text."""
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision)
        model_config = AutoConfig.from_pretrained(model_id, revision=model_revision)
    except OSError as error:
        raise _transformer_download_error(model_id, error) from error
    if not hasattr(model_config, 'hidden_size'):
        raise ValueError(f'{model_id!r} does not expose a Transformer hidden_size.')
    train_clean = preprocess_texts(train_texts, preprocessing)
    validation_clean = preprocess_texts(validation_texts, preprocessing)
    resolved_revision = getattr(model_config, '_commit_hash', None) or model_revision
    return {
        'tokenizer': tokenizer,
        'model_config': model_config,
        'train_clean': train_clean,
        'validation_clean': validation_clean,
        'manifest': {
            'backbone_download_source': 'huggingface_hub_internet',
            'internet_required_when_not_cached': 'true',
            'transformer_model_id': model_id,
            'transformer_requested_revision': model_revision,
            'transformer_resolved_revision': resolved_revision,
            'transformer_model_type': model_config.model_type,
            'transformer_hidden_size': model_config.hidden_size,
            'transformer_vocab_size': model_config.vocab_size,
            'transformer_max_position_embeddings': model_config.max_position_embeddings,
            'tokenizer_class': tokenizer.__class__.__name__,
        },
    }


def make_transformer_collate_fn(tokenizer, max_sequence_length: int):
    """Create a batch collator that applies the pretrained tokenizer consistently."""
    def collate(examples):
        """Tokenize one batch and return tensors accepted by AutoModel."""
        encoding = tokenizer(
            [example['text'] for example in examples],
            padding='max_length', truncation=True, max_length=max_sequence_length,
            return_tensors='pt',
        )
        encoding['targets'] = torch.tensor([example['target'] for example in examples], dtype=torch.long)
        return dict(encoding)
    return collate


def build_transformer_data_loader(texts, labels, tokenizer, config, shuffle: bool, seed: int):
    """Build a deterministic text DataLoader whose batches are tokenized together."""
    generator = torch.Generator()
    generator.manual_seed(seed)
    return torch.utils.data.DataLoader(
        TransformerTweetDataset(texts, labels), batch_size=config.batch_size,
        shuffle=shuffle, num_workers=0, pin_memory=device == 'cuda', generator=generator,
        collate_fn=make_transformer_collate_fn(tokenizer, config.max_sequence_length),
    )


def reset_pytorch_candidate_state(seed: int) -> None:
    """Clear releasable GPU memory and restore deterministic PyTorch candidate state."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def build_frozen_transformer_classifier(config: DistilBertClassifierConfig, resources: dict) -> torch.nn.Module:
    """Load a clean pretrained encoder from the Hub cache and freeze it for one candidate."""
    try:
        encoder = AutoModel.from_pretrained(config.model_id, revision=config.model_revision)
    except OSError as error:
        raise _transformer_download_error(config.model_id, error) from error
    return FrozenTransformerClassifier(
        encoder=encoder,
        hidden_size=resources['model_config'].hidden_size,
        classifier_hidden_units=config.classifier_hidden_units,
        dropout=config.dropout,
    ).to(device)


def _split_transformer_batch(batch: dict) -> tuple[dict, torch.Tensor]:
    """Move one tokenized batch to the selected device and separate the labels."""
    targets = batch['targets'].to(device)
    encoder_inputs = {name: value.to(device) for name, value in batch.items() if name != 'targets'}
    return encoder_inputs, targets


def train_transformer_epoch(model, data_loader, optimizer, loss_function) -> float:
    """Train only the dense head for one epoch and return sample-weighted loss."""
    model.train()
    loss_total = 0.0
    sample_count = 0
    for batch in data_loader:
        encoder_inputs, targets = _split_transformer_batch(batch)
        optimizer.zero_grad(set_to_none=True)
        logits = model(**encoder_inputs)
        loss = loss_function(logits, targets)
        loss.backward()
        optimizer.step()
        batch_size = len(targets)
        loss_total += float(loss.detach().item()) * batch_size
        sample_count += batch_size
    return loss_total / sample_count


def evaluate_transformer_batches(model, data_loader, loss_function):
    """Return loss, labels, probabilities and model-only inference time for one split."""
    model.eval()
    loss_total = 0.0
    sample_count = 0
    labels, probabilities = [], []
    model_inference_time_seconds = 0.0
    with torch.inference_mode():
        for batch in data_loader:
            encoder_inputs, targets = _split_transformer_batch(batch)
            inference_start = perf_counter()
            logits = model(**encoder_inputs)
            model_inference_time_seconds += perf_counter() - inference_start
            loss = loss_function(logits, targets)
            batch_size = len(targets)
            loss_total += float(loss.item()) * batch_size
            sample_count += batch_size
            labels.append(targets.cpu().numpy())
            probabilities.append(torch.softmax(logits, dim=1)[:, POSITIVE_LABEL].cpu().numpy())
    return loss_total / sample_count, np.concatenate(labels), np.concatenate(probabilities), model_inference_time_seconds


def transformer_parameter_counts(model) -> tuple[int, int]:
    """Return the trainable and total parameter counts for MLflow tracking."""
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in model.parameters())
    return int(trainable), int(total)


def fit_and_evaluate_distilbert_candidate(config, resources, train_labels, validation_labels, random_state: int) -> DistilBertCandidateResult:
    """Fit one frozen-DistilBERT dense head and evaluate it only on validation."""
    reset_pytorch_candidate_state(random_state)
    preparation_start = perf_counter()
    model = build_frozen_transformer_classifier(config, resources)
    train_loader = build_transformer_data_loader(resources['train_clean'], train_labels, resources['tokenizer'], config, True, random_state)
    validation_loader = build_transformer_data_loader(resources['validation_clean'], validation_labels, resources['tokenizer'], config, False, random_state)
    loss_function = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=config.learning_rate)
    input_preparation_time_seconds = perf_counter() - preparation_start

    history = {'loss': [], 'val_loss': [], 'val_accuracy': [], 'val_fbeta_negative': []}
    best_validation_loss = float('inf')
    best_state_dict = None
    best_epoch = 0
    stale_epoch_count = 0
    training_start = perf_counter()
    for epoch in range(1, config.max_epochs + 1):
        train_loss = train_transformer_epoch(model, train_loader, optimizer, loss_function)
        validation_loss, validation_truth, validation_probabilities, _ = evaluate_transformer_batches(model, validation_loader, loss_function)
        validation_predictions = labels_from_probabilities(validation_probabilities, threshold=DEFAULT_DECISION_THRESHOLD)
        validation_metrics = evaluate_binary_predictions(validation_truth, validation_predictions)
        history['loss'].append(float(train_loss))
        history['val_loss'].append(float(validation_loss))
        history['val_accuracy'].append(float(validation_metrics['accuracy']))
        history['val_fbeta_negative'].append(float(validation_metrics['fbeta_negative']))
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch
            best_state_dict = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale_epoch_count = 0
        else:
            stale_epoch_count += 1
            if stale_epoch_count >= config.early_stopping_patience:
                break
    train_time_seconds = perf_counter() - training_start
    if best_state_dict is None:
        raise RuntimeError('Frozen DistilBERT training did not produce a validation checkpoint.')
    model.load_state_dict(best_state_dict)
    _, validation_truth, validation_probabilities, validation_inference_time_seconds = evaluate_transformer_batches(model, validation_loader, loss_function)
    validation_predictions = labels_from_probabilities(validation_probabilities, threshold=DEFAULT_DECISION_THRESHOLD)
    trainable_parameter_count, total_parameter_count = transformer_parameter_counts(model)
    return DistilBertCandidateResult(
        config=config, model=model, validation_probabilities=validation_probabilities,
        validation_metrics=evaluate_binary_predictions(validation_truth, validation_predictions),
        history=history, train_time_seconds=train_time_seconds,
        input_preparation_time_seconds=input_preparation_time_seconds,
        validation_inference_time_seconds=validation_inference_time_seconds,
        best_epoch=best_epoch, trainable_parameter_count=trainable_parameter_count,
        total_parameter_count=total_parameter_count,
    )


def log_transformer_tokenizer_artifact(tokenizer) -> None:
    """Log the Hugging Face tokenizer files required to reproduce tokenization."""
    with tempfile.TemporaryDirectory(prefix='benchmark-artifact-') as temporary_directory:
        tokenizer.save_pretrained(temporary_directory)
        mlflow.log_artifacts(temporary_directory, artifact_path='transformer_tokenizer')


def log_transformer_checkpoint_artifact(model) -> None:
    """Log a CPU state_dict that serving code can reconstruct outside the notebook."""
    with tempfile.TemporaryDirectory(prefix='benchmark-artifact-') as temporary_directory:
        artifact_path = Path(temporary_directory) / 'transformer_checkpoint.pt'
        state_dict = {
            name: value.detach().cpu()
            for name, value in model.state_dict().items()
        }
        torch.save({'state_dict': state_dict}, artifact_path)
        mlflow.log_artifact(str(artifact_path))


## Data Import

Cette section charge le CSV Sentiment140, conserve le texte brut pour la traçabilité, remappe les labels en binaire et constitue un échantillon équilibré de taille maîtrisée avant toute modélisation.

In [ ]:
COLUMN_NAMES = ['target', 'id', 'datetime', 'query', 'username', 'tweet']
SAMPLE_SIZE = 125_000

# Import the raw Sentiment140 file with the `cp1252` encoding selected during EDA.
tweets = pd.read_csv(
    RAW_DATA_PATH,
    names=COLUMN_NAMES,
    header=None,
    encoding='cp1252'
)

# Keep the raw tweet text for traceability and remap the labels to a binary target for modeling.
tweets['tweet'] = tweets['tweet'].astype(str)
tweets['tweet_raw'] = tweets['tweet']
tweets['target_original'] = tweets['target']
tweets['target'] = tweets['target_original'].replace({0: 0, 4: 1})

# Build a balanced sample to keep the benchmark tractable while preserving class parity.
sample_per_class = SAMPLE_SIZE // tweets['target'].nunique()
data = (
    tweets.groupby('target', group_keys=False)
    .sample(n=sample_per_class, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
    .copy()
)

print('Imported rows in full dataset :', len(tweets))
print('Rows kept for benchmarking :', len(data))
print('Unique labels in sample :', sorted(data['target'].unique().tolist()))
data.head()

In [ ]:
# Compare the class balance between the full imported dataframe and the sampled benchmark dataframe.
distribution_summary = pd.DataFrame({
    'full_dataset': tweets['target'].value_counts(normalize=True).sort_index(),
    'sample': data['target'].value_counts(normalize=True).sort_index()
})
distribution_summary.index = distribution_summary.index.map({0: 'negative', 1: 'positive'})
(distribution_summary * 100).round(2)

## Train / Validation / Test Split

Cette section crée une séparation stratifiée et reproductible 60/20/20. Le train sert à ajuster les modèles, la validation à choisir configuration et seuil, et le test reste réservé à l'évaluation finale.

In [ ]:
TRAIN_RATIO = 0.6
VALIDATION_RATIO = 0.2
TEST_RATIO = 0.2

assert np.isclose(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO, 1.0), 'Split ratios must sum to 1.'

# First separate the training split from the remaining validation/test pool.
data_train, data_temp = train_test_split(
    data,
    train_size=TRAIN_RATIO,
    stratify=data['target'],
    random_state=RANDOM_STATE
)

# Then split the remaining pool equally to recover the requested 0.2 / 0.2 proportions.
validation_share_of_remaining = VALIDATION_RATIO / (VALIDATION_RATIO + TEST_RATIO)
data_val, data_test = train_test_split(
    data_temp,
    train_size=validation_share_of_remaining,
    stratify=data_temp['target'],
    random_state=RANDOM_STATE
)

# Reset indexes so later preprocessing steps can work on clean standalone dataframes.
data_train = data_train.reset_index(drop=True).copy()
data_val = data_val.reset_index(drop=True).copy()
data_test = data_test.reset_index(drop=True).copy()

print('Train shape :', data_train.shape)
print('Validation shape :', data_val.shape)
print('Test shape :', data_test.shape)

In [ ]:
# Verify both split sizes and class proportions before benchmarking.
split_distribution = pd.DataFrame({
    'train_rows': [len(data_train)],
    'validation_rows': [len(data_val)],
    'test_rows': [len(data_test)],
    'train_negative_pct': [round((data_train['target'] == 0).mean() * 100, 2)],
    'train_positive_pct': [round((data_train['target'] == 1).mean() * 100, 2)],
    'validation_negative_pct': [round((data_val['target'] == 0).mean() * 100, 2)],
    'validation_positive_pct': [round((data_val['target'] == 1).mean() * 100, 2)],
    'test_negative_pct': [round((data_test['target'] == 0).mean() * 100, 2)],
    'test_positive_pct': [round((data_test['target'] == 1).mean() * 100, 2)]
})
split_distribution

## Performance Metrics

Cette section définit les métriques communes des benchmarks, avec une attention particulière aux tweets négatifs assimilés au risque de bad buzz. Une cellule de contrôle illustre les calculs sur un exemple minimal.

In [ ]:
BADBUZZ_REFERENCE_VOLUME = 1000
NEGATIVE_LABEL = 0

def badbuzz_miss_rate(y_true, y_pred, negative_label=NEGATIVE_LABEL, reference_volume=BADBUZZ_REFERENCE_VOLUME):
    """Estimate how many truly negative tweets would be predicted as non-negative
    over a reference volume of negative tweets."""
    labels = sorted(pd.Index(y_true).append(pd.Index(y_pred)).unique().tolist())
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    label_to_index = {label: idx for idx, label in enumerate(labels)}

    if negative_label not in label_to_index:
        return np.nan

    neg_idx = label_to_index[negative_label]
    true_negatives = cm[neg_idx, neg_idx]
    false_positives = cm[neg_idx].sum() - true_negatives
    total_negative = true_negatives + false_positives

    if total_negative == 0:
        return np.nan

    return round(reference_volume * false_positives / total_negative)

def evaluate_predictions(y_true, y_pred, dataset_name='dataset', beta=3.0):
    """Compute the benchmark metrics used throughout the notebook for one set of predictions."""
    metrics_summary = pd.Series({
        'dataset': dataset_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_negative': precision_score(y_true, y_pred, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0),
        'recall_negative': recall_score(y_true, y_pred, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0),
        'fbeta_negative': fbeta_score(y_true, y_pred, beta=beta, pos_label=NEGATIVE_LABEL, average='binary', zero_division=0),
        f'badbuzz_miss_per_{BADBUZZ_REFERENCE_VOLUME}_negative_tweets': badbuzz_miss_rate(y_true, y_pred)
    })
    return metrics_summary.to_frame().T

def plot_confusion_matrix_from_predictions(y_true, y_pred, title='Confusion Matrix'):
    """Display a confusion matrix heatmap for a fitted model's predictions."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=['true_negative', 'true_positive'],
        columns=['pred_negative', 'pred_positive']
    )
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.show()

In [ ]:
# Small sanity check on handcrafted predictions so the metric definitions stay transparent.
y_true_example = pd.Series([0, 0, 0, 1, 1, 1])
y_pred_example = pd.Series([0, 1, 0, 1, 0, 1])

display(evaluate_predictions(y_true_example, y_pred_example, dataset_name='sanity_check'))
plot_confusion_matrix_from_predictions(y_true_example, y_pred_example, title='Performance metric sanity check')

## Generic Benchmark Protocol

Cette section définit les contrats, fonctions et callbacks partagés par toutes les familles du benchmark. Chaque famille conserve seulement ses opérations spécifiques ; le protocole expérimental reste identique : baseline, essais Optuna, résultats validation, sélection métier, refit du candidat retenu, réglage du seuil, évaluation test, puis reporting MLflow commun.


In [ ]:
@dataclass(frozen=True)
class BenchmarkDefinition:
    """Describe one benchmark family and its immutable search contract."""

    model_family: str
    model_name: str
    parent_run_name: str
    run_name_prefix: str
    study_name: str
    confusion_matrix_title: str
    baseline_config: Any
    optuna_trials: int
    optuna_timeout_seconds: int
    search_config: dict[str, Any]


@dataclass
class BenchmarkExecutionState:
    """Store the validation, selection and final outputs of one benchmark run."""

    validation_records: list[dict[str, Any]] = field(default_factory=list)
    configurations_by_name: dict[str, Any] = field(default_factory=dict)
    validation_results: pd.DataFrame | None = None
    optuna_results: pd.DataFrame | None = None
    threshold_results: pd.DataFrame | None = None
    selected_configuration_name: str | None = None
    selected_candidate: Any = None
    selected_threshold: float | None = None


@dataclass(frozen=True)
class CandidateTrackingExtras:
    """Hold model-family-specific values added to the shared MLflow tracking."""

    params: dict[str, Any] = field(default_factory=dict)
    metrics: dict[str, float] = field(default_factory=dict)
    record: dict[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class TestPrediction:
    """Return positive-class probabilities and measured test inference time."""

    probabilities: np.ndarray
    inference_time_seconds: float


@dataclass(frozen=True)
class TrackedCandidate:
    """Associate a fitted candidate with its validation record and MLflow run."""

    candidate: Any
    record: dict[str, Any]
    eligible: bool
    objective_value: float | None
    run_id: str


@dataclass(frozen=True)
class BenchmarkRunResult:
    """Expose the public outputs produced by the shared benchmark protocol."""

    validation_results: pd.DataFrame
    optuna_results: pd.DataFrame
    threshold_results: pd.DataFrame
    test_metrics: dict[str, float]
    summary: pd.DataFrame
    selected_candidate: Any
    selected_threshold: float


@dataclass(frozen=True)
class BenchmarkAdapter:
    """Provide the callbacks that specialize the shared protocol for one family."""

    parent_params: Callable[[], dict[str, Any]]
    log_parent_artifacts: Callable[[], None]
    suggest_configuration: Callable[[Any], Any]
    fit_candidate: Callable[[Any], Any]
    candidate_tracking_extras: Callable[[Any, str, bool, float | None], CandidateTrackingExtras]
    log_candidate_artifacts: Callable[[Any], None]
    selected_refit_metrics: Callable[[Any], dict[str, float]]
    log_selected_candidate_artifacts: Callable[[Any], None]
    predict_test: Callable[[Any], TestPrediction]
    log_final_artifacts: Callable[[Any, float], None]
    log_model: Callable[[Any], None]
    cleanup_candidate_state: Callable[[], None]


SELECTION_RULE_DESCRIPTION = (
    'validation_accuracy_gte_0.60_then_fbeta_negative_desc_then_'
    'badbuzz_miss_asc_then_accuracy_desc'
)
OPTUNA_OBJECTIVE_DESCRIPTION = 'fbeta_negative_if_accuracy_gte_0.60_else_-1.0'


def _candidate_objective(candidate) -> tuple[bool, float]:
    """Return the frozen Optuna objective for one validation candidate."""
    eligible = candidate.validation_metrics['accuracy'] >= MINIMUM_VALIDATION_ACCURACY
    objective_value = candidate.validation_metrics['fbeta_negative'] if eligible else -1.0
    return bool(eligible), float(objective_value)


def run_and_track_candidate(definition, adapter, configuration, validation_row_count, search_source, run_role, trial=None):
    """Fit and track one baseline or Optuna candidate in a nested MLflow run."""
    tags = {
        'run_role': run_role,
        'model_family': definition.model_family,
        'configuration_name': configuration.name,
    }
    if run_role == 'baseline':
        tags['is_baseline'] = 'true'
    if trial is not None:
        tags['optuna_trial_number'] = str(trial.number)

    with mlflow.start_run(
        run_name=f'{definition.run_name_prefix}-{configuration.name}',
        nested=True,
        tags=tags,
    ) as child_run:
        candidate = adapter.fit_candidate(configuration)
        eligible, objective_value = _candidate_objective(candidate)
        extras = adapter.candidate_tracking_extras(
            candidate, search_source, eligible,
            objective_value if trial is not None else None,
        )
        params = {
            **RUN_PROVENANCE_PARAMS,
            'model_family': definition.model_family,
            'model_name': definition.model_name,
            **configuration.as_mlflow_params(),
            'decision_threshold': DEFAULT_DECISION_THRESHOLD,
            **extras.params,
        }
        if trial is not None:
            params.update({
                'optuna_trial_number': trial.number,
                'optuna_objective_definition': OPTUNA_OBJECTIVE_DESCRIPTION,
            })
        mlflow.log_params(params)
        metrics = {
            **{f'validation_{name}': float(value) for name, value in candidate.validation_metrics.items()},
            'training_time_seconds': candidate.train_time_seconds,
            'validation_inference_time_seconds': candidate.validation_inference_time_seconds,
            'validation_inference_ms_per_tweet': (
                candidate.validation_inference_time_seconds * 1000 / validation_row_count
            ),
            **extras.metrics,
        }
        if trial is not None:
            metrics['optuna_objective'] = objective_value
        mlflow.log_metrics(metrics)
        adapter.log_candidate_artifacts(candidate)
        record = candidate.summary_record()
        record.update({
            'search_source': search_source,
            'optuna_trial_number': np.nan if trial is None else trial.number,
            'optuna_objective': np.nan if trial is None else objective_value,
            'mlflow_run_id': child_run.info.run_id,
            **extras.record,
        })

    return TrackedCandidate(
        candidate=candidate, record=record, eligible=eligible,
        objective_value=objective_value if trial is not None else None,
        run_id=child_run.info.run_id,
    )


def run_baseline_phase(definition, adapter, state, validation_row_count):
    """Evaluate, track and release the fixed baseline before Optuna starts."""
    configuration = definition.baseline_config
    tracked = run_and_track_candidate(
        definition, adapter, configuration, validation_row_count,
        search_source='baseline', run_role='baseline',
    )
    state.validation_records.append(tracked.record)
    state.configurations_by_name[configuration.name] = configuration
    del tracked
    adapter.cleanup_candidate_state()


def run_optuna_phase(definition, adapter, state, validation_row_count):
    """Run the current seeded, sequential TPE search and track every trial."""
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=PROJECT_SEED),
        pruner=optuna.pruners.NopPruner(),
        study_name=definition.study_name,
    )

    def objective(trial):
        """Fit, track and return the frozen validation objective for one trial."""
        configuration = adapter.suggest_configuration(trial)
        tracked = run_and_track_candidate(
            definition, adapter, configuration, validation_row_count,
            search_source='optuna_tpe', run_role='optuna_trial', trial=trial,
        )
        state.validation_records.append(tracked.record)
        state.configurations_by_name[configuration.name] = configuration
        trial.set_user_attr('configuration_name', configuration.name)
        trial.set_user_attr('validation_accuracy', tracked.candidate.validation_metrics['accuracy'])
        trial.set_user_attr('validation_eligible', tracked.eligible)
        trial.set_user_attr('mlflow_run_id', tracked.run_id)
        objective_value = float(tracked.objective_value)
        del tracked
        adapter.cleanup_candidate_state()
        return objective_value

    study.optimize(
        objective, n_trials=definition.optuna_trials,
        timeout=definition.optuna_timeout_seconds, n_jobs=1,
    )
    return study


def finalize_validation_search(definition, state, study, parent_start_time):
    """Log common validation and Optuna artifacts for the model family."""
    completed_trials = [
        trial for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
    ]
    state.validation_results = pd.DataFrame(state.validation_records)
    state.optuna_results = state.validation_results.loc[
        state.validation_results['search_source'] == 'optuna_tpe'
    ].copy()
    log_dataframe_artifact(state.validation_results, 'validation_configuration_results.csv')
    log_dataframe_artifact(state.optuna_results, 'optuna_trial_results.csv')
    log_json_artifact(definition.search_config, 'optuna_search_config.json')
    mlflow.log_metrics({
        'optuna_completed_trial_count': len(completed_trials),
        'optuna_search_time_seconds': perf_counter() - parent_start_time,
    })


def select_and_refit_candidate(definition, adapter, state):
    """Select by the business rule and refit only the selected configuration."""
    selected_row = select_by_validation_rule(state.validation_results)
    selected_name = str(selected_row['configuration_name'])
    selected_configuration = state.configurations_by_name[selected_name]
    selected_candidate = adapter.fit_candidate(selected_configuration)
    state.selected_configuration_name = selected_name
    state.selected_candidate = selected_candidate
    mlflow.set_tags({
        'selection_status': 'configuration_selected',
        'selected_configuration': selected_name,
        'selected_for_test': 'true',
    })
    mlflow.log_params({
        'selected_configuration': selected_name,
        'configuration_selection_rule': SELECTION_RULE_DESCRIPTION,
        **{f'selected_{name}': value for name, value in selected_configuration.as_mlflow_params().items()},
    })
    mlflow.log_metrics({
        **{f'selected_refit_validation_{name}': float(value) for name, value in selected_candidate.validation_metrics.items()},
        'selected_refit_training_time_seconds': selected_candidate.train_time_seconds,
        'selected_refit_validation_inference_time_seconds': selected_candidate.validation_inference_time_seconds,
        **adapter.selected_refit_metrics(selected_candidate),
    })
    adapter.log_selected_candidate_artifacts(selected_candidate)


def tune_selected_threshold(definition, state, validation_labels):
    """Tune and freeze the decision threshold using validation only."""
    with mlflow.start_run(
        run_name=f'{definition.run_name_prefix}-threshold-tuning-{state.selected_configuration_name}',
        nested=True,
        tags={
            'run_role': 'threshold_tuning',
            'model_family': definition.model_family,
            'selected_configuration': state.selected_configuration_name,
        },
    ):
        start_time = perf_counter()
        state.threshold_results = search_thresholds(
            validation_labels, state.selected_candidate.validation_probabilities,
            default_threshold=DEFAULT_DECISION_THRESHOLD,
        )
        selected_row = select_by_validation_rule(state.threshold_results)
        default_row = state.threshold_results.loc[
            state.threshold_results['is_default_threshold']
        ].iloc[0]
        state.selected_threshold = float(selected_row['threshold'])
        mlflow.log_params({
            **RUN_PROVENANCE_PARAMS,
            'selected_configuration': state.selected_configuration_name,
            'default_decision_threshold': DEFAULT_DECISION_THRESHOLD,
            'selected_decision_threshold': state.selected_threshold,
            'threshold_candidate_count': len(state.threshold_results),
        })
        mlflow.log_metrics({
            **{f'validation_default_threshold_{name}': float(default_row[name]) for name in state.selected_candidate.validation_metrics},
            **{f'validation_tuned_threshold_{name}': float(selected_row[name]) for name in state.selected_candidate.validation_metrics},
            'threshold_tuning_time_seconds': perf_counter() - start_time,
        })
        log_dataframe_artifact(state.threshold_results, 'threshold_search_results.csv')
    return selected_row


def evaluate_test_and_finalize(definition, adapter, state, test_labels, test_row_count, parent_start_time, selected_threshold_row):
    """Evaluate test once and log common final artifacts and summary."""
    test_prediction = adapter.predict_test(state.selected_candidate)
    test_predictions = labels_from_probabilities(test_prediction.probabilities, state.selected_threshold)
    test_metrics = evaluate_binary_predictions(test_labels, test_predictions)
    test_report = classification_report(
        test_labels, test_predictions, labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
        target_names=['negative', 'positive'], output_dict=True, zero_division=0,
    )
    matrix = confusion_matrix(test_labels, test_predictions, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
    matrix_frame = pd.DataFrame(
        matrix, index=['true_negative', 'true_positive'],
        columns=['pred_negative', 'pred_positive'],
    )
    figure, axis = plt.subplots(figsize=(6, 5))
    sns.heatmap(matrix_frame, annot=True, fmt='d', cmap='Blues', ax=axis)
    axis.set_title(definition.confusion_matrix_title)
    axis.set_xlabel('Predicted label')
    axis.set_ylabel('True label')
    plt.show()
    log_figure_artifact(figure, 'test_confusion_matrix.png')
    plt.close(figure)
    mlflow.log_metrics({
        **{f'test_{name}': float(value) for name, value in test_metrics.items()},
        'test_inference_time_seconds': test_prediction.inference_time_seconds,
        'test_inference_ms_per_tweet': test_prediction.inference_time_seconds * 1000 / test_row_count,
        'model_family_total_time_seconds': perf_counter() - parent_start_time,
    })
    mlflow.log_params({'frozen_decision_threshold': state.selected_threshold})
    log_dataframe_artifact(pd.DataFrame([{**{'dataset': 'test'}, **test_metrics}]), 'test_metrics.csv')
    log_json_artifact({'threshold': state.selected_threshold}, 'decision_threshold.json')
    log_json_artifact(test_report, 'test_classification_report.json')
    adapter.log_final_artifacts(state.selected_candidate, state.selected_threshold)
    adapter.log_model(state.selected_candidate)
    mlflow.set_tags({
        'selection_status': 'test_evaluated',
        'threshold_status': 'frozen_after_validation_tuning',
    })
    summary = pd.DataFrame([{
        'model_family': definition.model_family,
        'model_name': definition.model_name,
        'selected_configuration': state.selected_configuration_name,
        'frozen_threshold': state.selected_threshold,
        **{f'validation_{name}': float(selected_threshold_row[name]) for name in state.selected_candidate.validation_metrics},
        **{f'test_{name}': float(value) for name, value in test_metrics.items()},
        'training_time_seconds': state.selected_candidate.train_time_seconds,
        'test_inference_time_seconds': test_prediction.inference_time_seconds,
    }])
    return test_metrics, summary


def run_benchmark_protocol(definition, adapter, tracking_config, validation_labels, test_labels, validation_row_count, test_row_count):
    """Execute the shared baseline-to-test protocol for one model family."""
    state = BenchmarkExecutionState()
    with mlflow.start_run(
        experiment_id=tracking_config.experiment_id,
        run_name=definition.parent_run_name,
        tags={
            'run_role': 'model_family_parent',
            'model_family': definition.model_family,
            'model_name': definition.model_name,
            'selection_status': 'in_progress',
        },
    ):
        parent_start_time = perf_counter()
        mlflow.log_params(adapter.parent_params())
        adapter.log_parent_artifacts()
        run_baseline_phase(definition, adapter, state, validation_row_count)
        study = run_optuna_phase(definition, adapter, state, validation_row_count)
        finalize_validation_search(definition, state, study, parent_start_time)
        select_and_refit_candidate(definition, adapter, state)
        threshold_row = tune_selected_threshold(definition, state, validation_labels)
        test_metrics, summary = evaluate_test_and_finalize(
            definition, adapter, state, test_labels, test_row_count,
            parent_start_time, threshold_row,
        )
    return BenchmarkRunResult(
        validation_results=state.validation_results, optuna_results=state.optuna_results,
        threshold_results=state.threshold_results, test_metrics=test_metrics,
        summary=summary, selected_candidate=state.selected_candidate,
        selected_threshold=state.selected_threshold,
    )


## TF-IDF + Logistic Regression

Cette section exécute la première famille de référence en suivant le protocole commun : baseline fixe, recherche Optuna sur train/validation, sélection métier, réglage du seuil et une seule évaluation test.

This first benchmark family follows the shared protocol: one fixed baseline,
then a bounded Optuna search on the fixed training/validation splits. The
business selection rule and threshold are chosen on validation only; the
untouched test split is evaluated once at the end.


In [ ]:
# Edit these switches to change social_text_v1 without changing preprocessing code.
# `replace_numbers` and `remove_digits` are alternatives; the same is true for
# `encode_sentiment_punctuation` and `remove_punctuation`.
SOCIAL_TEXT_V1_PARAMETERS = {
    'normalize_whitespace': True,
    'decode_html_entities': True,
    'expand_contractions': True,
    'replace_urls': True,
    'replace_emails': True,
    'replace_mentions': True,
    'expand_hashtags': True,
    'demojize_emojis': True,
    'replace_ascii_emoticons': False,
    'reduce_elongated_tokens': False,
    'replace_numbers': True,
    'encode_sentiment_punctuation': True,
    'remove_digits': False,
    'remove_punctuation': False,
}

TFIDF_LOGREG_PREPROCESSING_PROFILES = {
    'whitespace_v1': TextPreprocessingConfig.whitespace_v1(),
    'social_text_v1': TextPreprocessingConfig.social_text_v1(**SOCIAL_TEXT_V1_PARAMETERS),
}
TFIDF_LOGREG_BASELINE_CONFIG = default_baseline_configuration(
    TFIDF_LOGREG_PREPROCESSING_PROFILES['whitespace_v1']
)
TFIDF_LOGREG_OPTUNA_TRIALS = 20
TFIDF_LOGREG_OPTUNA_TIMEOUT_SECONDS = 60 * 60
TFIDF_LOGREG_MODEL_FAMILY = 'tfidf_logistic_regression'
TFIDF_LOGREG_MODEL_NAME = 'TF-IDF + Logistic Regression'

pd.DataFrame([
    {
        'configuration_name': TFIDF_LOGREG_BASELINE_CONFIG.name,
        'search_source': 'baseline',
        **TFIDF_LOGREG_BASELINE_CONFIG.as_mlflow_params(),
    },
    {
        'configuration_name': 'optuna_search_space',
        'search_source': 'optuna_tpe',
        'preprocessing_profiles': ', '.join(TFIDF_LOGREG_PREPROCESSING_PROFILES),
        'optuna_trials': TFIDF_LOGREG_OPTUNA_TRIALS,
        'optuna_timeout_seconds': TFIDF_LOGREG_OPTUNA_TIMEOUT_SECONDS,
    },
])


In [ ]:
# def _no_op(*args, **kwargs):
#     """Provide an explicit no-op callback for adapters without extra work."""
#     return None


# def build_tfidf_adapter() -> BenchmarkAdapter:
#     """Build callbacks for the TF-IDF plus Logistic Regression family."""
#     def parent_params():
#         """Return the unchanged parent-run metadata for the TF-IDF family."""
#         return {
#             **RUN_PROVENANCE_PARAMS,
#             'model_family': TFIDF_LOGREG_MODEL_FAMILY,
#             'model_name': TFIDF_LOGREG_MODEL_NAME,
#             'benchmark_split': 'stratified_60_20_20',
#             'training_rows': len(data_train),
#             'validation_rows': len(data_val),
#             'test_rows': len(data_test),
#             'optuna_n_trials_requested': TFIDF_LOGREG_OPTUNA_TRIALS,
#             'optuna_timeout_seconds': TFIDF_LOGREG_OPTUNA_TIMEOUT_SECONDS,
#             'optuna_sampler': 'TPESampler',
#             'selection_minimum_validation_accuracy': MINIMUM_VALIDATION_ACCURACY,
#         }

#     def fit_candidate(configuration):
#         """Fit one TF-IDF configuration on train and evaluate it on validation."""
#         return fit_and_evaluate_candidate(
#             configuration,
#             train_texts=data_train['tweet'],
#             train_labels=data_train['target'],
#             validation_texts=data_val['tweet'],
#             validation_labels=data_val['target'],
#             random_state=PROJECT_SEED,
#         )

#     def candidate_tracking_extras(candidate, search_source, eligible, objective_value):
#         """Expose TF-IDF vocabulary size and the classifier seed for MLflow."""
#         vocabulary_size = len(candidate.pipeline.named_steps['tfidf'].vocabulary_)
#         return CandidateTrackingExtras(
#             params={'classifier_random_state': PROJECT_SEED},
#             metrics={'tfidf_vocabulary_size': float(vocabulary_size)},
#             record={'tfidf_vocabulary_size': vocabulary_size},
#         )

#     def selected_refit_metrics(candidate):
#         """Return the TF-IDF metric specific to the selected refit."""
#         vocabulary_size = len(candidate.pipeline.named_steps['tfidf'].vocabulary_)
#         return {'selected_refit_tfidf_vocabulary_size': float(vocabulary_size)}

#     def predict_test(candidate):
#         """Predict positive probabilities on the untouched TF-IDF test split."""
#         inference_start = perf_counter()
#         probabilities = probabilities_for_positive_class(candidate.pipeline, data_test['tweet'])
#         return TestPrediction(
#             probabilities=probabilities,
#             inference_time_seconds=perf_counter() - inference_start,
#         )

#     return BenchmarkAdapter(
#         parent_params=parent_params,
#         log_parent_artifacts=_no_op,
#         suggest_configuration=lambda trial: suggest_optuna_configuration(
#             trial,
#             preprocessing_profiles=TFIDF_LOGREG_PREPROCESSING_PROFILES,
#         ),
#         fit_candidate=fit_candidate,
#         candidate_tracking_extras=candidate_tracking_extras,
#         log_candidate_artifacts=_no_op,
#         selected_refit_metrics=selected_refit_metrics,
#         log_selected_candidate_artifacts=_no_op,
#         predict_test=predict_test,
#         log_final_artifacts=_no_op,
#         log_model=lambda candidate: mlflow.sklearn.log_model(
#             candidate.pipeline,
#             name='selected_model',
#         ),
#         cleanup_candidate_state=_no_op,
#     )


# tfidf_definition = BenchmarkDefinition(
#     model_family=TFIDF_LOGREG_MODEL_FAMILY,
#     model_name=TFIDF_LOGREG_MODEL_NAME,
#     parent_run_name='tfidf-logreg-benchmark',
#     run_name_prefix='tfidf-logreg',
#     study_name='tfidf_logreg_validation_search',
#     confusion_matrix_title='TF-IDF + Logistic Regression - test confusion matrix',
#     baseline_config=TFIDF_LOGREG_BASELINE_CONFIG,
#     optuna_trials=TFIDF_LOGREG_OPTUNA_TRIALS,
#     optuna_timeout_seconds=TFIDF_LOGREG_OPTUNA_TIMEOUT_SECONDS,
#     search_config={
#         'search_strategy': 'baseline_plus_optuna_tpe',
#         'n_trials_requested': TFIDF_LOGREG_OPTUNA_TRIALS,
#         'timeout_seconds': TFIDF_LOGREG_OPTUNA_TIMEOUT_SECONDS,
#         'sampler': 'TPESampler',
#         'sampler_seed': PROJECT_SEED,
#         'pruner': 'NopPruner',
#         'preprocessing_profiles': {
#             name: config.as_mlflow_params()
#             for name, config in TFIDF_LOGREG_PREPROCESSING_PROFILES.items()
#         },
#         'search_space': {
#             'tfidf_ngram_max': [1, 2],
#             'tfidf_min_df': [2, 5, 10],
#             'tfidf_max_df': [0.95, 1.0],
#             'tfidf_max_features': [75_000, 120_000],
#             'classifier_c_log_range': [0.1, 5.0],
#             'classifier_penalty': ['l1', 'l2'],
#             'classifier_solver': 'liblinear',
#         },
#         'objective_definition': OPTUNA_OBJECTIVE_DESCRIPTION,
#         'final_selection_rule': SELECTION_RULE_DESCRIPTION,
#     },
# )

# tfidf_result = run_benchmark_protocol(
#     definition=tfidf_definition,
#     adapter=build_tfidf_adapter(),
#     tracking_config=tracking_config,
#     validation_labels=data_val['target'],
#     test_labels=data_test['target'],
#     validation_row_count=len(data_val),
#     test_row_count=len(data_test),
# )

# TFIDF_LOGREG_VALIDATION_RESULTS = tfidf_result.validation_results
# TFIDF_LOGREG_OPTUNA_RESULTS = tfidf_result.optuna_results
# TFIDF_LOGREG_THRESHOLD_RESULTS = tfidf_result.threshold_results
# TFIDF_LOGREG_TEST_METRICS = tfidf_result.test_metrics
# TFIDF_LOGREG_BENCHMARK_SUMMARY = tfidf_result.summary

# display(TFIDF_LOGREG_VALIDATION_RESULTS.sort_values(
#     by=['fbeta_negative', 'badbuzz_miss_per_1000_negative_tweets', 'accuracy'],
#     ascending=[False, True, False],
# ))
# display(TFIDF_LOGREG_BENCHMARK_SUMMARY)


## GloVe + LSTM

Cette section exécute, seulement lorsqu'elle est activée, la famille neuronale GloVe + LSTM. Elle prépare les embeddings attachés à Kaggle, entraîne baseline et candidates Optuna, puis applique le même protocole de sélection que TF-IDF.

This neural family preserves the historical GloVe Twitter 27B (50d) baseline,
but follows the shared benchmark protocol: train/validation selection, frozen
threshold, then one test evaluation. Attach a Kaggle dataset containing
\`glove.twitter.27B.50d.txt\` before enabling the section.


In [ ]:
# Set this to True only after a Kaggle dataset containing the GloVe file is attached.
# Leave the path as None for automatic discovery under /kaggle/input.
RUN_GLOVE_LSTM_BENCHMARK = True
GLOVE_EMBEDDING_PATH = os.environ.get('GLOVE_EMBEDDING_PATH') or None

# This profile remains fully explicit and can be changed without editing utilities.
# The two extra switches reproduce useful behavior from the historical tweet cleaner.
GLOVE_LSTM_SOCIAL_TEXT_PARAMETERS = {
    **SOCIAL_TEXT_V1_DEFAULT_PARAMETERS,
    'replace_ascii_emoticons': True,
    'reduce_elongated_tokens': True,
}
GLOVE_LSTM_PREPROCESSING = TextPreprocessingConfig.social_text_v1(
    **GLOVE_LSTM_SOCIAL_TEXT_PARAMETERS
)

GLOVE_LSTM_MODEL_FAMILY = 'glove_lstm'
GLOVE_LSTM_MODEL_NAME = 'GloVe Twitter 27B (50d) + LSTM'
GLOVE_LSTM_MAX_VOCABULARY_SIZE = 75_000
GLOVE_LSTM_OPTUNA_TRIALS = 8
GLOVE_LSTM_OPTUNA_TIMEOUT_SECONDS = 2 * 60 * 60
GLOVE_LSTM_SEARCH_SPACE_VERSION = 'glove_lstm_v1'

# Historical baseline: frozen GloVe 50d -> LSTM(196, dropout=0.2) -> sigmoid.
GLOVE_LSTM_BASELINE_CONFIG = GloveLstmConfig(
    name='baseline_glove50_lstm196',
    preprocessing=GLOVE_LSTM_PREPROCESSING,
    vocabulary_size=75_000,
    max_sequence_length=64,
    lstm_units=196,
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=64,
    max_epochs=10,
    early_stopping_patience=2,
)

GLOVE_LSTM_BENCHMARK_SUMMARY = pd.DataFrame()
pd.DataFrame([
    {
        'configuration_name': GLOVE_LSTM_BASELINE_CONFIG.name,
        'search_source': 'baseline',
        **GLOVE_LSTM_BASELINE_CONFIG.as_mlflow_params(),
    },
    {
        'configuration_name': 'optuna_search_space',
        'search_source': 'optuna_tpe',
        'optuna_trials': GLOVE_LSTM_OPTUNA_TRIALS,
        'optuna_timeout_seconds': GLOVE_LSTM_OPTUNA_TIMEOUT_SECONDS,
        'search_space_version': GLOVE_LSTM_SEARCH_SPACE_VERSION,
    },
])


In [ ]:
# def build_glove_lstm_adapter(glove_resources, glove_asset_params) -> BenchmarkAdapter:
#     """Build callbacks for the GloVe Twitter 50d plus LSTM family."""
#     def parent_params():
#         """Return the unchanged parent-run metadata for the GloVe family."""
#         return {
#             **RUN_PROVENANCE_PARAMS,
#             **glove_asset_params,
#             'model_family': GLOVE_LSTM_MODEL_FAMILY,
#             'model_name': GLOVE_LSTM_MODEL_NAME,
#             'benchmark_split': 'stratified_60_20_20',
#             'training_rows': len(data_train),
#             'validation_rows': len(data_val),
#             'test_rows': len(data_test),
#             'optuna_n_trials_requested': GLOVE_LSTM_OPTUNA_TRIALS,
#             'optuna_timeout_seconds': GLOVE_LSTM_OPTUNA_TIMEOUT_SECONDS,
#             'optuna_sampler': 'TPESampler',
#             'selection_minimum_validation_accuracy': MINIMUM_VALIDATION_ACCURACY,
#             'tokenizer_train_unique_token_count': len(glove_resources['tokenizer'].word_index),
#             'glove_vector_cache_count': len(glove_resources['vectors']),
#         }

#     def suggest_configuration(trial):
#         """Sample one bounded GloVe/LSTM configuration from the existing search space."""
#         return GloveLstmConfig(
#             name=f'optuna_trial_{trial.number:03d}',
#             preprocessing=GLOVE_LSTM_PREPROCESSING,
#             vocabulary_size=trial.suggest_categorical(
#                 'tokenizer_vocabulary_size',
#                 [50_000, 75_000],
#             ),
#             max_sequence_length=trial.suggest_categorical(
#                 'sequence_max_length',
#                 [48, 64],
#             ),
#             lstm_units=trial.suggest_categorical('lstm_units', [128, 196]),
#             dropout=trial.suggest_categorical('lstm_dropout', [0.1, 0.2, 0.3]),
#             learning_rate=trial.suggest_categorical(
#                 'optimizer_learning_rate',
#                 [3e-4, 1e-3],
#             ),
#             batch_size=trial.suggest_categorical('batch_size', [64, 128]),
#             max_epochs=GLOVE_LSTM_BASELINE_CONFIG.max_epochs,
#             early_stopping_patience=GLOVE_LSTM_BASELINE_CONFIG.early_stopping_patience,
#         )

#     def fit_candidate(configuration):
#         """Fit one GloVe/LSTM configuration with shared prepared resources."""
#         return fit_and_evaluate_glove_lstm_candidate(
#             configuration,
#             glove_resources,
#             data_train['target'],
#             data_val['target'],
#             PROJECT_SEED,
#         )

#     def candidate_tracking_extras(candidate, search_source, eligible, objective_value):
#         """Expose GloVe coverage and neural training details for MLflow."""
#         return CandidateTrackingExtras(
#             params={
#                 **glove_asset_params,
#                 'search_source': search_source,
#                 'validation_eligible': str(eligible).lower(),
#             },
#             metrics={
#                 **candidate.coverage,
#                 'training_epoch_count': float(len(candidate.history.get('loss', []))),
#                 'best_epoch': float(candidate.best_epoch),
#                 'input_preparation_time_seconds': candidate.input_preparation_time_seconds,
#             },
#         )

#     def selected_refit_metrics(candidate):
#         """Return GloVe coverage and epoch details for the selected refit."""
#         return {
#             **{
#                 f'selected_refit_{name}': float(value)
#                 for name, value in candidate.coverage.items()
#             },
#             'selected_refit_best_epoch': float(candidate.best_epoch),
#         }

#     def predict_test(candidate):
#         """Prepare test sequences and predict their positive probabilities."""
#         test_clean = preprocess_texts(data_test['tweet'], candidate.config.preprocessing)
#         test_sequences = texts_to_padded_sequences(
#             glove_resources['tokenizer'],
#             test_clean,
#             candidate.config.vocabulary_size,
#             candidate.config.max_sequence_length,
#         )
#         inference_start = perf_counter()
#         probabilities = candidate.model.predict(
#             test_sequences,
#             batch_size=candidate.config.batch_size,
#             verbose=0,
#         ).reshape(-1)
#         return TestPrediction(
#             probabilities=probabilities,
#             inference_time_seconds=perf_counter() - inference_start,
#         )

#     def log_final_artifacts(candidate, selected_threshold):
#         """Log tokenizer, sequence, coverage and inference-contract artifacts."""
#         log_json_artifact(
#             tokenizer_artifact_payload(
#                 glove_resources['tokenizer'],
#                 candidate.config.vocabulary_size,
#             ),
#             'tokenizer.json',
#         )
#         log_json_artifact(
#             candidate.config.preprocessing.as_mlflow_params(),
#             'preprocessing_config.json',
#         )
#         sequence_config = {
#             'max_sequence_length': candidate.config.max_sequence_length,
#             'padding': 'post',
#             'truncating': 'post',
#             'vocabulary_size': candidate.effective_vocabulary_size,
#             'oov_token': glove_resources['tokenizer'].oov_token,
#             'filters': TOKENIZER_FILTERS_PRESERVING_UNDERSCORE,
#         }
#         log_json_artifact(sequence_config, 'sequence_config.json')
#         log_json_artifact(candidate.coverage, 'glove_coverage.json')
#         log_json_artifact({
#             'preprocessing_artifact': 'preprocessing_config.json',
#             'tokenizer_artifact': 'tokenizer.json',
#             'sequence_artifact': 'sequence_config.json',
#             'model_artifact': 'selected_model',
#             'probability_label': POSITIVE_LABEL,
#             'frozen_threshold': selected_threshold,
#         }, 'inference_contract.json')

#     return BenchmarkAdapter(
#         parent_params=parent_params,
#         log_parent_artifacts=lambda: log_json_artifact(
#             GLOVE_LSTM_ASSET_MANIFEST,
#             'glove_asset_manifest.json',
#         ),
#         suggest_configuration=suggest_configuration,
#         fit_candidate=fit_candidate,
#         candidate_tracking_extras=candidate_tracking_extras,
#         log_candidate_artifacts=lambda candidate: log_neural_learning_history(candidate.history),
#         selected_refit_metrics=selected_refit_metrics,
#         log_selected_candidate_artifacts=lambda candidate: log_neural_learning_history(candidate.history),
#         predict_test=predict_test,
#         log_final_artifacts=log_final_artifacts,
#         log_model=lambda candidate: mlflow.tensorflow.log_model(
#             candidate.model,
#             name='selected_model',
#         ),
#         cleanup_candidate_state=lambda: reset_tensorflow_candidate_state(PROJECT_SEED),
#     )


# if not RUN_GLOVE_LSTM_BENCHMARK:
#     print(
#         'GloVe + LSTM is disabled. Attach a Kaggle dataset containing '
#         f'{GLOVE_TWITTER_27B_FILENAME}, then set RUN_GLOVE_LSTM_BENCHMARK = True.'
#     )
# else:
#     glove_asset_path = resolve_glove_embedding_path(GLOVE_EMBEDDING_PATH)
#     glove_resources = prepare_glove_lstm_resources(
#         train_texts=data_train['tweet'],
#         validation_texts=data_val['tweet'],
#         preprocessing=GLOVE_LSTM_PREPROCESSING,
#         asset_path=glove_asset_path,
#         maximum_vocabulary_size=GLOVE_LSTM_MAX_VOCABULARY_SIZE,
#         embedding_dimension=GLOVE_TWITTER_27B_DIMENSION,
#     )
#     GLOVE_LSTM_ASSET_MANIFEST = glove_resources['manifest']
#     glove_asset_params = {
#         f'glove_asset_{name}': value
#         for name, value in GLOVE_LSTM_ASSET_MANIFEST.items()
#     }
#     glove_definition = BenchmarkDefinition(
#         model_family=GLOVE_LSTM_MODEL_FAMILY,
#         model_name=GLOVE_LSTM_MODEL_NAME,
#         parent_run_name='glove-lstm-benchmark',
#         run_name_prefix='glove-lstm',
#         study_name='glove_lstm_validation_search',
#         confusion_matrix_title='GloVe + LSTM - test confusion matrix',
#         baseline_config=GLOVE_LSTM_BASELINE_CONFIG,
#         optuna_trials=GLOVE_LSTM_OPTUNA_TRIALS,
#         optuna_timeout_seconds=GLOVE_LSTM_OPTUNA_TIMEOUT_SECONDS,
#         search_config={
#             'search_strategy': 'baseline_plus_optuna_tpe',
#             'n_trials_requested': GLOVE_LSTM_OPTUNA_TRIALS,
#             'timeout_seconds': GLOVE_LSTM_OPTUNA_TIMEOUT_SECONDS,
#             'sampler': 'TPESampler',
#             'sampler_seed': PROJECT_SEED,
#             'pruner': 'NopPruner',
#             'baseline_configuration': GLOVE_LSTM_BASELINE_CONFIG.as_mlflow_params(),
#             'search_space': {
#                 'tokenizer_vocabulary_size': [50_000, 75_000],
#                 'sequence_max_length': [48, 64],
#                 'lstm_units': [128, 196],
#                 'lstm_dropout': [0.1, 0.2, 0.3],
#                 'optimizer_learning_rate': [3e-4, 1e-3],
#                 'batch_size': [64, 128],
#             },
#             'objective_definition': OPTUNA_OBJECTIVE_DESCRIPTION,
#             'final_selection_rule': SELECTION_RULE_DESCRIPTION,
#         },
#     )
#     glove_adapter = build_glove_lstm_adapter(glove_resources, glove_asset_params)
#     glove_result = run_benchmark_protocol(
#         definition=glove_definition,
#         adapter=glove_adapter,
#         tracking_config=tracking_config,
#         validation_labels=data_val['target'],
#         test_labels=data_test['target'],
#         validation_row_count=len(data_val),
#         test_row_count=len(data_test),
#     )
#     GLOVE_LSTM_VALIDATION_RESULTS = glove_result.validation_results
#     GLOVE_LSTM_OPTUNA_RESULTS = glove_result.optuna_results
#     GLOVE_LSTM_THRESHOLD_RESULTS = glove_result.threshold_results
#     GLOVE_LSTM_TEST_METRICS = glove_result.test_metrics
#     GLOVE_LSTM_BENCHMARK_SUMMARY = glove_result.summary
#     display(GLOVE_LSTM_VALIDATION_RESULTS.sort_values(
#         by=['fbeta_negative', 'badbuzz_miss_per_1000_negative_tweets', 'accuracy'],
#         ascending=[False, True, False],
#     ))
#     display(GLOVE_LSTM_BENCHMARK_SUMMARY)
#     del glove_result
#     glove_adapter.cleanup_candidate_state()


## Word2Vec + GRU

Cette section ajoute une famille neuronale où Word2Vec est appris une seule fois, uniquement sur les tweets de train prétraités. Les vecteurs sont ensuite gelés pendant la recherche de la GRU. La baseline, les essais Optuna, la sélection sur validation, le réglage du seuil et l'évaluation test suivent le protocole partagé ; le jeu de test reste donc intact jusqu'à la dernière étape.


In [ ]:
# Activate this family explicitly after the preceding benchmark cells have run successfully.
RUN_WORD2VEC_GRU_BENCHMARK = True

# The social-text preprocessing stays explicit and can be adjusted without editing utilities.
WORD2VEC_GRU_SOCIAL_TEXT_PARAMETERS = {
    **SOCIAL_TEXT_V1_DEFAULT_PARAMETERS,
    'replace_ascii_emoticons': True,
    'reduce_elongated_tokens': True,
}
WORD2VEC_GRU_PREPROCESSING = TextPreprocessingConfig.social_text_v1(
    **WORD2VEC_GRU_SOCIAL_TEXT_PARAMETERS
)

# Word2Vec is a fixed train-only asset for the entire benchmark run, not an Optuna variable.
WORD2VEC_TRAINING_CONFIG = Word2VecTrainingConfig(
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    negative=10,
    epochs=10,
    seed=PROJECT_SEED,
    workers=1,
)

WORD2VEC_GRU_MODEL_FAMILY = 'word2vec_gru'
WORD2VEC_GRU_MODEL_NAME = 'Word2Vec (train-only, 100d) + GRU'
WORD2VEC_GRU_MAX_VOCABULARY_SIZE = 75_000
WORD2VEC_GRU_OPTUNA_TRIALS = 8
WORD2VEC_GRU_OPTUNA_TIMEOUT_SECONDS = 2 * 60 * 60
WORD2VEC_GRU_SEARCH_SPACE_VERSION = 'word2vec_gru_v1'

# Baseline: frozen train-only Word2Vec(100d) -> GRU(196, dropout=0.2) -> sigmoid.
WORD2VEC_GRU_BASELINE_CONFIG = Word2VecGruConfig(
    name='baseline_word2vec100_gru196',
    preprocessing=WORD2VEC_GRU_PREPROCESSING,
    vocabulary_size=75_000,
    max_sequence_length=64,
    gru_units=196,
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=64,
    max_epochs=10,
    early_stopping_patience=2,
    embedding_dimension=WORD2VEC_TRAINING_CONFIG.vector_size,
)

WORD2VEC_GRU_VALIDATION_RESULTS = pd.DataFrame()
WORD2VEC_GRU_OPTUNA_RESULTS = pd.DataFrame()
WORD2VEC_GRU_THRESHOLD_RESULTS = pd.DataFrame()
WORD2VEC_GRU_TEST_METRICS = {}
WORD2VEC_GRU_BENCHMARK_SUMMARY = pd.DataFrame()

pd.DataFrame([
    {
        'configuration_name': WORD2VEC_GRU_BASELINE_CONFIG.name,
        'search_source': 'baseline',
        **WORD2VEC_GRU_BASELINE_CONFIG.as_mlflow_params(),
        **WORD2VEC_TRAINING_CONFIG.as_mlflow_params(),
    },
    {
        'configuration_name': 'optuna_search_space',
        'search_source': 'optuna_tpe',
        'optuna_trials': WORD2VEC_GRU_OPTUNA_TRIALS,
        'optuna_timeout_seconds': WORD2VEC_GRU_OPTUNA_TIMEOUT_SECONDS,
        'search_space_version': WORD2VEC_GRU_SEARCH_SPACE_VERSION,
    },
])


In [ ]:
# def build_word2vec_gru_adapter(word2vec_resources, word2vec_asset_params) -> BenchmarkAdapter:
#     """Build callbacks for the train-only Word2Vec plus frozen-embedding GRU family."""

#     def parent_params():
#         """Return parent-run metadata for the Word2Vec and GRU benchmark."""
#         return {
#             **RUN_PROVENANCE_PARAMS,
#             **WORD2VEC_TRAINING_CONFIG.as_mlflow_params(),
#             **word2vec_asset_params,
#             'model_family': WORD2VEC_GRU_MODEL_FAMILY,
#             'model_name': WORD2VEC_GRU_MODEL_NAME,
#             'benchmark_split': 'stratified_60_20_20',
#             'training_rows': len(data_train),
#             'validation_rows': len(data_val),
#             'test_rows': len(data_test),
#             'optuna_n_trials_requested': WORD2VEC_GRU_OPTUNA_TRIALS,
#             'optuna_timeout_seconds': WORD2VEC_GRU_OPTUNA_TIMEOUT_SECONDS,
#             'optuna_sampler': 'TPESampler',
#             'selection_minimum_validation_accuracy': MINIMUM_VALIDATION_ACCURACY,
#             'tokenizer_train_unique_token_count': len(word2vec_resources['tokenizer'].word_index),
#         }

#     def log_parent_artifacts():
#         """Persist the fixed train-only Word2Vec asset and its provenance once."""
#         log_json_artifact(
#             WORD2VEC_TRAINING_CONFIG.as_mlflow_params(),
#             'word2vec_training_config.json',
#         )
#         log_json_artifact(WORD2VEC_GRU_MANIFEST, 'word2vec_manifest.json')
#         log_word2vec_model_artifact(word2vec_resources['word2vec_model'])

#     def suggest_configuration(trial):
#         """Sample one bounded GRU configuration while keeping Word2Vec fixed."""
#         return Word2VecGruConfig(
#             name=f'optuna_trial_{trial.number:03d}',
#             preprocessing=WORD2VEC_GRU_PREPROCESSING,
#             vocabulary_size=trial.suggest_categorical(
#                 'tokenizer_vocabulary_size', [50_000, 75_000],
#             ),
#             max_sequence_length=trial.suggest_categorical(
#                 'sequence_max_length', [48, 64],
#             ),
#             gru_units=trial.suggest_categorical('gru_units', [128, 196]),
#             dropout=trial.suggest_categorical('gru_dropout', [0.1, 0.2, 0.3]),
#             learning_rate=trial.suggest_categorical(
#                 'optimizer_learning_rate', [3e-4, 1e-3],
#             ),
#             batch_size=trial.suggest_categorical('batch_size', [64, 128]),
#             max_epochs=WORD2VEC_GRU_BASELINE_CONFIG.max_epochs,
#             early_stopping_patience=WORD2VEC_GRU_BASELINE_CONFIG.early_stopping_patience,
#             embedding_dimension=WORD2VEC_TRAINING_CONFIG.vector_size,
#         )

#     def fit_candidate(configuration):
#         """Fit one GRU candidate using the shared train-only Word2Vec resources."""
#         return fit_and_evaluate_word2vec_gru_candidate(
#             configuration,
#             word2vec_resources,
#             data_train['target'],
#             data_val['target'],
#             PROJECT_SEED,
#         )

#     def candidate_tracking_extras(candidate, search_source, eligible, objective_value):
#         """Expose Word2Vec coverage and neural training details for MLflow."""
#         return CandidateTrackingExtras(
#             params={
#                 **word2vec_asset_params,
#                 'search_source': search_source,
#                 'validation_eligible': str(eligible).lower(),
#             },
#             metrics={
#                 **candidate.coverage,
#                 'training_epoch_count': float(len(candidate.history.get('loss', []))),
#                 'best_epoch': float(candidate.best_epoch),
#                 'input_preparation_time_seconds': candidate.input_preparation_time_seconds,
#             },
#         )

#     def selected_refit_metrics(candidate):
#         """Return Word2Vec coverage and epoch details for the selected refit."""
#         return {
#             **{
#                 f'selected_refit_{name}': float(value)
#                 for name, value in candidate.coverage.items()
#             },
#             'selected_refit_best_epoch': float(candidate.best_epoch),
#         }

#     def predict_test(candidate):
#         """Preprocess test tweets and obtain probabilities after validation is complete."""
#         test_clean = preprocess_texts(data_test['tweet'], candidate.config.preprocessing)
#         test_sequences = texts_to_padded_sequences(
#             word2vec_resources['tokenizer'],
#             test_clean,
#             candidate.config.vocabulary_size,
#             candidate.config.max_sequence_length,
#         )
#         inference_start = perf_counter()
#         probabilities = candidate.model.predict(
#             test_sequences,
#             batch_size=candidate.config.batch_size,
#             verbose=0,
#         ).reshape(-1)
#         return TestPrediction(
#             probabilities=probabilities,
#             inference_time_seconds=perf_counter() - inference_start,
#         )

#     def log_final_artifacts(candidate, selected_threshold):
#         """Log the artifacts required to reproduce selected-model inference."""
#         log_json_artifact(
#             tokenizer_artifact_payload(
#                 word2vec_resources['tokenizer'],
#                 candidate.config.vocabulary_size,
#             ),
#             'tokenizer.json',
#         )
#         log_json_artifact(
#             candidate.config.preprocessing.as_mlflow_params(),
#             'preprocessing_config.json',
#         )
#         log_json_artifact({
#             'max_sequence_length': candidate.config.max_sequence_length,
#             'padding': 'post',
#             'truncating': 'post',
#             'vocabulary_size': candidate.effective_vocabulary_size,
#             'oov_token': word2vec_resources['tokenizer'].oov_token,
#             'filters': TOKENIZER_FILTERS_PRESERVING_UNDERSCORE,
#         }, 'sequence_config.json')
#         log_json_artifact(candidate.coverage, 'word2vec_coverage.json')
#         log_json_artifact({
#             'preprocessing_artifact': 'preprocessing_config.json',
#             'tokenizer_artifact': 'tokenizer.json',
#             'sequence_artifact': 'sequence_config.json',
#             'word2vec_artifact': 'word2vec.model',
#             'model_artifact': 'selected_model',
#             'probability_label': POSITIVE_LABEL,
#             'frozen_threshold': selected_threshold,
#         }, 'inference_contract.json')

#     return BenchmarkAdapter(
#         parent_params=parent_params,
#         log_parent_artifacts=log_parent_artifacts,
#         suggest_configuration=suggest_configuration,
#         fit_candidate=fit_candidate,
#         candidate_tracking_extras=candidate_tracking_extras,
#         log_candidate_artifacts=lambda candidate: log_neural_learning_history(candidate.history),
#         selected_refit_metrics=selected_refit_metrics,
#         log_selected_candidate_artifacts=lambda candidate: log_neural_learning_history(candidate.history),
#         predict_test=predict_test,
#         log_final_artifacts=log_final_artifacts,
#         log_model=lambda candidate: mlflow.tensorflow.log_model(
#             candidate.model,
#             name='selected_model',
#         ),
#         cleanup_candidate_state=lambda: reset_tensorflow_candidate_state(PROJECT_SEED),
#     )


# if not RUN_WORD2VEC_GRU_BENCHMARK:
#     print(
#         'Word2Vec + GRU is disabled. Set RUN_WORD2VEC_GRU_BENCHMARK = True '
#         'to learn Word2Vec on train and run this benchmark family.'
#     )
# else:
#     word2vec_resources = prepare_word2vec_gru_resources(
#         train_texts=data_train['tweet'],
#         validation_texts=data_val['tweet'],
#         preprocessing=WORD2VEC_GRU_PREPROCESSING,
#         word2vec_config=WORD2VEC_TRAINING_CONFIG,
#         maximum_vocabulary_size=WORD2VEC_GRU_MAX_VOCABULARY_SIZE,
#     )
#     WORD2VEC_GRU_MANIFEST = word2vec_resources['manifest']
#     word2vec_asset_params = {
#         'word2vec_training_sentence_count': WORD2VEC_GRU_MANIFEST['word2vec_sentence_count'],
#         'word2vec_vocabulary_count': WORD2VEC_GRU_MANIFEST['word2vec_vocabulary_count'],
#         'word2vec_training_time_seconds': WORD2VEC_GRU_MANIFEST['word2vec_training_time_seconds'],
#     }
#     word2vec_definition = BenchmarkDefinition(
#         model_family=WORD2VEC_GRU_MODEL_FAMILY,
#         model_name=WORD2VEC_GRU_MODEL_NAME,
#         parent_run_name='word2vec-gru-benchmark',
#         run_name_prefix='word2vec-gru',
#         study_name='word2vec_gru_validation_search',
#         confusion_matrix_title='Word2Vec + GRU - test confusion matrix',
#         baseline_config=WORD2VEC_GRU_BASELINE_CONFIG,
#         optuna_trials=WORD2VEC_GRU_OPTUNA_TRIALS,
#         optuna_timeout_seconds=WORD2VEC_GRU_OPTUNA_TIMEOUT_SECONDS,
#         search_config={
#             'search_strategy': 'baseline_plus_optuna_tpe',
#             'n_trials_requested': WORD2VEC_GRU_OPTUNA_TRIALS,
#             'timeout_seconds': WORD2VEC_GRU_OPTUNA_TIMEOUT_SECONDS,
#             'sampler': 'TPESampler',
#             'sampler_seed': PROJECT_SEED,
#             'pruner': 'NopPruner',
#             'search_space_version': WORD2VEC_GRU_SEARCH_SPACE_VERSION,
#             'word2vec_training_configuration': WORD2VEC_TRAINING_CONFIG.as_mlflow_params(),
#             'baseline_configuration': WORD2VEC_GRU_BASELINE_CONFIG.as_mlflow_params(),
#             'search_space': {
#                 'tokenizer_vocabulary_size': [50_000, 75_000],
#                 'sequence_max_length': [48, 64],
#                 'gru_units': [128, 196],
#                 'gru_dropout': [0.1, 0.2, 0.3],
#                 'optimizer_learning_rate': [3e-4, 1e-3],
#                 'batch_size': [64, 128],
#             },
#             'objective_definition': OPTUNA_OBJECTIVE_DESCRIPTION,
#             'final_selection_rule': SELECTION_RULE_DESCRIPTION,
#         },
#     )
#     word2vec_adapter = build_word2vec_gru_adapter(
#         word2vec_resources,
#         word2vec_asset_params,
#     )
#     word2vec_result = run_benchmark_protocol(
#         definition=word2vec_definition,
#         adapter=word2vec_adapter,
#         tracking_config=tracking_config,
#         validation_labels=data_val['target'],
#         test_labels=data_test['target'],
#         validation_row_count=len(data_val),
#         test_row_count=len(data_test),
#     )
#     WORD2VEC_GRU_VALIDATION_RESULTS = word2vec_result.validation_results
#     WORD2VEC_GRU_OPTUNA_RESULTS = word2vec_result.optuna_results
#     WORD2VEC_GRU_THRESHOLD_RESULTS = word2vec_result.threshold_results
#     WORD2VEC_GRU_TEST_METRICS = word2vec_result.test_metrics
#     WORD2VEC_GRU_BENCHMARK_SUMMARY = word2vec_result.summary
#     display(WORD2VEC_GRU_VALIDATION_RESULTS.sort_values(
#         by=['fbeta_negative', 'badbuzz_miss_per_1000_negative_tweets', 'accuracy'],
#         ascending=[False, True, False],
#     ))
#     display(WORD2VEC_GRU_BENCHMARK_SUMMARY)
#     del word2vec_result
#     word2vec_adapter.cleanup_candidate_state()


## DistilBERT gelé + tête dense

Cette dernière famille du benchmark reprend l'approche historique BERT + DNN avec `distilbert-base-uncased` : un encodeur Transformer pré-entraîné, gelé, puis une tête dense entraînable. Le modèle et son tokenizer sont téléchargés directement depuis Hugging Face au premier lancement Kaggle : activer Internet dans les réglages du notebook avant d'exécuter cette section. Le protocole partagé conserve la sélection sur validation, le réglage du seuil, puis l'unique évaluation finale sur test.


In [ ]:
# Enable Kaggle Internet before activating this benchmark: the backbone is downloaded from Hugging Face.
RUN_DISTILBERT_FROZEN_CLASSIFIER_BENCHMARK = True
DISTILBERT_MODEL_ID = 'distilbert-base-uncased'
DISTILBERT_MODEL_REVISION = 'main'

# Preserve tweet semantics while leaving digits and punctuation available to the Transformer tokenizer.
DISTILBERT_SOCIAL_TEXT_PARAMETERS = {
    **SOCIAL_TEXT_V1_DEFAULT_PARAMETERS,
    'replace_ascii_emoticons': True,
    'reduce_elongated_tokens': True,
    'encode_sentiment_punctuation': False,
    'remove_digits': False,
    'remove_punctuation': False,
}
DISTILBERT_PREPROCESSING = TextPreprocessingConfig.social_text_v1(**DISTILBERT_SOCIAL_TEXT_PARAMETERS)

DISTILBERT_MODEL_FAMILY = 'distilbert_frozen_dense_classifier'
DISTILBERT_MODEL_NAME = 'DistilBERT frozen encoder + dense classifier'
DISTILBERT_OPTUNA_TRIALS = 8
DISTILBERT_OPTUNA_TIMEOUT_SECONDS = 2 * 60 * 60
DISTILBERT_SEARCH_SPACE_VERSION = 'distilbert_frozen_head_v1'

# Historical baseline: frozen DistilBERT -> masked mean pooling -> Dense(256, ReLU, dropout=0.3) -> 2 logits.
DISTILBERT_BASELINE_CONFIG = DistilBertClassifierConfig(
    name='baseline_distilbert_frozen_head256',
    model_id=DISTILBERT_MODEL_ID,
    model_revision=DISTILBERT_MODEL_REVISION,
    preprocessing=DISTILBERT_PREPROCESSING,
    max_sequence_length=64,
    classifier_hidden_units=256,
    dropout=0.3,
    learning_rate=1e-3,
    batch_size=128,
    max_epochs=8,
    early_stopping_patience=2,
)

DISTILBERT_VALIDATION_RESULTS = pd.DataFrame()
DISTILBERT_OPTUNA_RESULTS = pd.DataFrame()
DISTILBERT_THRESHOLD_RESULTS = pd.DataFrame()
DISTILBERT_TEST_METRICS = {}
DISTILBERT_BENCHMARK_SUMMARY = pd.DataFrame()

pd.DataFrame([
    {'configuration_name': DISTILBERT_BASELINE_CONFIG.name, 'search_source': 'baseline', **DISTILBERT_BASELINE_CONFIG.as_mlflow_params()},
    {
        'configuration_name': 'optuna_search_space', 'search_source': 'optuna_tpe',
        'optuna_trials': DISTILBERT_OPTUNA_TRIALS,
        'optuna_timeout_seconds': DISTILBERT_OPTUNA_TIMEOUT_SECONDS,
        'search_space_version': DISTILBERT_SEARCH_SPACE_VERSION,
    },
])


In [ ]:
def build_distilbert_adapter(distilbert_resources, distilbert_asset_params) -> BenchmarkAdapter:
    """Build callbacks for the Internet-downloaded frozen DistilBERT dense classifier."""

    def parent_params():
        """Return parent-run metadata for the DistilBERT benchmark family."""
        return {
            **RUN_PROVENANCE_PARAMS, **distilbert_asset_params,
            'model_family': DISTILBERT_MODEL_FAMILY,
            'model_name': DISTILBERT_MODEL_NAME,
            'benchmark_split': 'stratified_60_20_20',
            'training_rows': len(data_train), 'validation_rows': len(data_val), 'test_rows': len(data_test),
            'optuna_n_trials_requested': DISTILBERT_OPTUNA_TRIALS,
            'optuna_timeout_seconds': DISTILBERT_OPTUNA_TIMEOUT_SECONDS,
            'optuna_sampler': 'TPESampler',
            'selection_minimum_validation_accuracy': MINIMUM_VALIDATION_ACCURACY,
            'pytorch_device': device,
        }

    def log_parent_artifacts():
        """Log model provenance and architecture metadata without copying Hub weights."""
        log_json_artifact(DISTILBERT_MANIFEST, 'transformer_manifest.json')
        log_json_artifact(distilbert_resources['model_config'].to_dict(), 'transformer_model_config.json')

    def suggest_configuration(trial):
        """Sample a bounded dense head while keeping DistilBERT frozen."""
        return DistilBertClassifierConfig(
            name=f'optuna_trial_{trial.number:03d}',
            model_id=DISTILBERT_MODEL_ID, model_revision=DISTILBERT_MODEL_REVISION,
            preprocessing=DISTILBERT_PREPROCESSING,
            max_sequence_length=trial.suggest_categorical('sequence_max_length', [64, 96]),
            classifier_hidden_units=trial.suggest_categorical('classifier_hidden_units', [128, 256, 384]),
            dropout=trial.suggest_categorical('classifier_dropout', [0.1, 0.3, 0.5]),
            learning_rate=trial.suggest_categorical('optimizer_learning_rate', [1e-4, 3e-4, 1e-3]),
            batch_size=trial.suggest_categorical('batch_size', [64, 128]),
            max_epochs=DISTILBERT_BASELINE_CONFIG.max_epochs,
            early_stopping_patience=DISTILBERT_BASELINE_CONFIG.early_stopping_patience,
        )

    def fit_candidate(configuration):
        """Fit one dense head using shared text and a fresh frozen backbone."""
        return fit_and_evaluate_distilbert_candidate(
            configuration, distilbert_resources, data_train['target'], data_val['target'], PROJECT_SEED,
        )

    def candidate_tracking_extras(candidate, search_source, eligible, objective_value):
        """Expose Transformer parameter counts and epoch details for each child run."""
        return CandidateTrackingExtras(
            params={**distilbert_asset_params, 'search_source': search_source, 'validation_eligible': str(eligible).lower()},
            metrics={
                'trainable_parameter_count': float(candidate.trainable_parameter_count),
                'total_parameter_count': float(candidate.total_parameter_count),
                'training_epoch_count': float(len(candidate.history.get('loss', []))),
                'best_epoch': float(candidate.best_epoch),
                'input_preparation_time_seconds': candidate.input_preparation_time_seconds,
            },
        )

    def selected_refit_metrics(candidate):
        """Return model-size and checkpoint details for the selected refit."""
        return {
            'selected_refit_trainable_parameter_count': float(candidate.trainable_parameter_count),
            'selected_refit_total_parameter_count': float(candidate.total_parameter_count),
            'selected_refit_best_epoch': float(candidate.best_epoch),
        }

    def predict_test(candidate):
        """Tokenize test only after validation selection and return positive probabilities."""
        test_clean = preprocess_texts(data_test['tweet'], candidate.config.preprocessing)
        test_loader = build_transformer_data_loader(
            test_clean, data_test['target'], distilbert_resources['tokenizer'], candidate.config, False, PROJECT_SEED,
        )
        _, _, probabilities, inference_time_seconds = evaluate_transformer_batches(
            candidate.model, test_loader, torch.nn.CrossEntropyLoss(),
        )
        return TestPrediction(probabilities=probabilities, inference_time_seconds=inference_time_seconds)

    def log_final_artifacts(candidate, selected_threshold):
        """Log tokenizer and the reconstruction contract for the selected classifier."""
        log_transformer_tokenizer_artifact(distilbert_resources['tokenizer'])
        log_transformer_checkpoint_artifact(candidate.model)
        log_json_artifact(candidate.config.preprocessing.as_mlflow_params(), 'preprocessing_config.json')
        log_json_artifact({
            'max_sequence_length': candidate.config.max_sequence_length,
            'padding': 'max_length', 'truncation': True, 'positive_label': POSITIVE_LABEL,
        }, 'sequence_config.json')
        log_json_artifact({
            'transformer_model_id': candidate.config.model_id,
            'transformer_requested_revision': candidate.config.model_revision,
            'transformer_resolved_revision': DISTILBERT_MANIFEST['transformer_resolved_revision'],
            'backbone_trainable': candidate.config.backbone_trainable,
            'pooling': 'masked_mean_last_hidden_state',
            'classifier_hidden_units': candidate.config.classifier_hidden_units,
            'classifier_dropout': candidate.config.dropout,
            'tokenizer_artifact_directory': 'transformer_tokenizer',
            'checkpoint_artifact': 'transformer_checkpoint.pt',
            'model_artifact': 'selected_model', 'probability_label': POSITIVE_LABEL,
            'frozen_threshold': selected_threshold,
        }, 'inference_contract.json')

    def log_model(candidate):
        """Persist the selected PyTorch wrapper after moving it off GPU memory."""
        candidate.model.to('cpu')
        mlflow.pytorch.log_model(candidate.model, name='selected_model')

    return BenchmarkAdapter(
        parent_params=parent_params, log_parent_artifacts=log_parent_artifacts,
        suggest_configuration=suggest_configuration, fit_candidate=fit_candidate,
        candidate_tracking_extras=candidate_tracking_extras,
        log_candidate_artifacts=lambda candidate: log_neural_learning_history(candidate.history),
        selected_refit_metrics=selected_refit_metrics,
        log_selected_candidate_artifacts=lambda candidate: log_neural_learning_history(candidate.history),
        predict_test=predict_test, log_final_artifacts=log_final_artifacts,
        log_model=log_model, cleanup_candidate_state=lambda: reset_pytorch_candidate_state(PROJECT_SEED),
    )


if not RUN_DISTILBERT_FROZEN_CLASSIFIER_BENCHMARK:
    print(
        'DistilBERT frozen classifier is disabled. Enable Kaggle Internet, then set '
        'RUN_DISTILBERT_FROZEN_CLASSIFIER_BENCHMARK = True to run it.'
    )
else:
    distilbert_resources = prepare_distilbert_resources(
        data_train['tweet'], data_val['tweet'], DISTILBERT_PREPROCESSING,
        DISTILBERT_MODEL_ID, DISTILBERT_MODEL_REVISION,
    )
    DISTILBERT_MANIFEST = distilbert_resources['manifest']
    distilbert_asset_params = dict(DISTILBERT_MANIFEST)
    distilbert_definition = BenchmarkDefinition(
        model_family=DISTILBERT_MODEL_FAMILY, model_name=DISTILBERT_MODEL_NAME,
        parent_run_name='distilbert-frozen-classifier-benchmark',
        run_name_prefix='distilbert-frozen', study_name='distilbert_frozen_validation_search',
        confusion_matrix_title='DistilBERT frozen classifier - test confusion matrix',
        baseline_config=DISTILBERT_BASELINE_CONFIG,
        optuna_trials=DISTILBERT_OPTUNA_TRIALS,
        optuna_timeout_seconds=DISTILBERT_OPTUNA_TIMEOUT_SECONDS,
        search_config={
            'search_strategy': 'baseline_plus_optuna_tpe',
            'n_trials_requested': DISTILBERT_OPTUNA_TRIALS,
            'timeout_seconds': DISTILBERT_OPTUNA_TIMEOUT_SECONDS,
            'sampler': 'TPESampler', 'sampler_seed': PROJECT_SEED, 'pruner': 'NopPruner',
            'search_space_version': DISTILBERT_SEARCH_SPACE_VERSION,
            'backbone': {
                'model_id': DISTILBERT_MODEL_ID, 'requested_revision': DISTILBERT_MODEL_REVISION,
                'download_source': 'huggingface_hub_internet', 'backbone_trainable': False,
                'pooling': 'masked_mean_last_hidden_state',
            },
            'baseline_configuration': DISTILBERT_BASELINE_CONFIG.as_mlflow_params(),
            'search_space': {
                'sequence_max_length': [64, 96], 'classifier_hidden_units': [128, 256, 384],
                'classifier_dropout': [0.1, 0.3, 0.5],
                'optimizer_learning_rate': [1e-4, 3e-4, 1e-3], 'batch_size': [64, 128],
            },
            'objective_definition': OPTUNA_OBJECTIVE_DESCRIPTION,
            'final_selection_rule': SELECTION_RULE_DESCRIPTION,
        },
    )
    distilbert_adapter = build_distilbert_adapter(distilbert_resources, distilbert_asset_params)
    distilbert_result = run_benchmark_protocol(
        definition=distilbert_definition, adapter=distilbert_adapter,
        tracking_config=tracking_config, validation_labels=data_val['target'], test_labels=data_test['target'],
        validation_row_count=len(data_val), test_row_count=len(data_test),
    )
    DISTILBERT_VALIDATION_RESULTS = distilbert_result.validation_results
    DISTILBERT_OPTUNA_RESULTS = distilbert_result.optuna_results
    DISTILBERT_THRESHOLD_RESULTS = distilbert_result.threshold_results
    DISTILBERT_TEST_METRICS = distilbert_result.test_metrics
    DISTILBERT_BENCHMARK_SUMMARY = distilbert_result.summary
    display(DISTILBERT_VALIDATION_RESULTS.sort_values(
        by=['fbeta_negative', 'badbuzz_miss_per_1000_negative_tweets'], ascending=[False, True],
    ))
    display(DISTILBERT_BENCHMARK_SUMMARY)
    del distilbert_result
    distilbert_adapter.cleanup_candidate_state()
